In [1]:
import json
import os
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore", category=RuntimeWarning)  # 忽略 NaN 触发的 numpy RuntimeWarning
import functools

print = functools.partial(print, flush=True)  # 全局 print 实时 flush，防止内核异常退出时丢失尾部日志

try:
    from joblib import Parallel, delayed
except ImportError:
    Parallel = None
    delayed = None

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

# ================= LLM 环境变量配置 =================
# 请在这里填写你的 LLM API 配置；如果不配置或留空，会自动跳过 LLM 生成。
# 使用 setdefault 是为了不覆盖已经在系统环境变量中设置的值。
os.environ.setdefault("LLM_API_KEY", "sk-5e75c2834752498ca9999f92fbd3810f")
os.environ.setdefault("LLM_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1")
os.environ.setdefault("LLM_MODEL", "glm-5.2")
os.environ.setdefault("LLM_N", "10")
os.environ.setdefault("LLM_TIMEOUT", "900")  # glm-5.2 为推理模型，响应慢，默认 300s 易读超时
os.environ.setdefault("LLM_JSON_MODE", "1")

# ================= Vocabulary (src/vocab.py) =================
"""
RL + RPN 词表（比赛版本）

数据来源：
- 平台日频因子库 bigalpha_2026_factorlib（主要字段）
- 平台 1 分钟行情库 bigalpha_2026_stock_bar1m（原始分钟字段 + 日聚合算子）

所有字段名在 RPN 表达式中保持和 DataFrame 列名一致。
"""

from typing import Dict, List, Optional, Tuple


class Vocabulary:
    """RPN token 词表 + 动作掩码辅助函数。"""

    # 1) 平台日频因子库字段
    BASE_FIELDS: List[str] = [
        # 价格/收益/成交量
        "close", "volume", "amount", "turn", "change_ratio", "daily_return",
        # 动量/反转/波动
        "momentum_5", "reversal_5", "volatility_5",
        # 市值/估值
        "total_market_cap", "float_market_cap",
        "pe_ttm", "pb", "ps_ttm",
        # 技术指标
        "sma_20", "ema_20",
        "macd_diff_12_26_9", "macd_dea_12_26_9", "macd_hist_12_26_9",
        "rsi_12",
        "kdj_k_9_3_3", "kdj_d_9_3_3",
        "bias_20", "cci_14", "atr_14",
        # 财务
        "roe_avg_ttm", "roa_avg_ttm",
        "gross_profit_rate_ttm", "net_profit_rate_ttm",
        "debt_to_asset_lf", "current_ratio_lf",
        # 资金流
        "netflow_amount_main", "netflow_amount_rate_main", "net_active_buy_amount_main",
        # 风险/其它
        "beta_000300SH_22", "list_days",
    ]

    # 2) 1 分钟表聚合出的高频日频特征
    HF_FIELDS: List[str] = [
        "realized_vol",          # 日内已实现波动
        "intraday_return",       # (close - open) / open
        "morning_return",        # 9:30-10:30 收益
        "afternoon_return",      # 14:00-收盘 收益
        "vwap_dev",              # (close - vwap) / vwap
        "ob_imbalance_l5",       # 五档盘口买卖量 imbalance
        "spread_l1_mean",        # 一档价差均值
        "quote_intensity_bid",   # 买方委托笔数强度
        "quote_intensity_ask",   # 卖方委托笔数强度
    ]

    # 3) 1 分钟表原始字段（ curated 子集，用于分钟级 RPN）
    RAW_MINUTE_FIELDS: List[str] = (
        ["m_pre_close", "m_open", "m_high", "m_low", "m_close",
         "m_volume", "m_amount", "m_deal_number"]
        + [f"m_ask_price{i}" for i in range(1, 6)]
        + [f"m_bid_price{i}" for i in range(1, 6)]
        + [f"m_ask_volume{i}" for i in range(1, 6)]
        + [f"m_bid_volume{i}" for i in range(1, 6)]
        + [f"m_ask_num_orders{i}" for i in range(1, 6)]
        + [f"m_bid_num_orders{i}" for i in range(1, 6)]
    )

    # 4) 常量
    CONSTANTS: List[str] = ["-10", "-5", "-3", "-2", "-1", "0", "1", "2", "3", "5", "10", "20", "60"]

    # 5) 算子：token -> (arity, category)
    OPERATORS: Dict[str, Tuple[int, str]] = {
        # 一元算子
        "ABS": (1, "unary"),
        "SIGN": (1, "unary"),
        "NEG": (1, "unary"),
        "LOG": (1, "unary"),
        # 截面算子
        "Cs_Rank": (1, "cross_section"),
        "Cs_Zscore": (1, "cross_section"),
        # 日频时序算子
        "Ts_Mean_5": (1, "time_series"),
        "Ts_Mean_10": (1, "time_series"),
        "Ts_Mean_20": (1, "time_series"),
        "Ts_Std_5": (1, "time_series"),
        "Ts_Std_20": (1, "time_series"),
        "Ts_Delay_1": (1, "time_series"),
        "Ts_Delay_5": (1, "time_series"),
        "Ts_Rank_5": (1, "time_series"),
        "Ts_Rank_20": (1, "time_series"),
        "Ts_Max_5": (1, "time_series"),
        "Ts_Min_5": (1, "time_series"),
        "Ts_Sum_5": (1, "time_series"),
        "Ts_Zscore_20": (1, "time_series"),
        # 分钟频时序算子（窗口以分钟数计）
        "Ms_Mean_5": (1, "minute_series"),
        "Ms_Mean_30": (1, "minute_series"),
        "Ms_Std_5": (1, "minute_series"),
        "Ms_Std_30": (1, "minute_series"),
        "Ms_Delay_1": (1, "minute_series"),
        "Ms_Delay_5": (1, "minute_series"),
        "Ms_Rank_5": (1, "minute_series"),
        "Ms_Rank_30": (1, "minute_series"),
        "Ms_Max_5": (1, "minute_series"),
        "Ms_Min_5": (1, "minute_series"),
        "Ms_Sum_5": (1, "minute_series"),
        # 分钟 -> 日聚合算子
        "Day_Last": (1, "daily_agg"),
        "Day_Mean": (1, "daily_agg"),
        "Day_Std": (1, "daily_agg"),
        "Day_Max": (1, "daily_agg"),
        "Day_Min": (1, "daily_agg"),
        "Day_Sum": (1, "daily_agg"),
        # 二元算子
        "ADD": (2, "binary"),
        "SUB": (2, "binary"),
        "MUL": (2, "binary"),
        "DIV": (2, "binary"),
        "MAX": (2, "binary"),
        "MIN": (2, "binary"),
        # 三元算子
        "If_Else": (3, "ternary"),
        # 特殊
        "BEG": (0, "special"),
        "SEP": (0, "special"),
    }

    def __init__(self, field_subset: Optional[List[str]] = None, include_minute_ops: bool = True):
        if field_subset is not None:
            self.fields = list(field_subset)
        else:
            self.fields = self.BASE_FIELDS + self.HF_FIELDS + self.RAW_MINUTE_FIELDS
        self.constants = self.CONSTANTS
        # include_minute_ops=False 时剔除 Ms_*/Day_* 分钟算子：纯日频面板下这些算子
        # 必然求值失败（InvalidExpressionError），曾让 490/507 个随机候选出生即死。
        self.operators = [
            op for op in self.OPERATORS
            if include_minute_ops or self.OPERATORS[op][1] not in ("minute_series", "daily_agg")
        ]
        self.tokens = ["BEG"] + self.fields + self.constants + self.operators + ["SEP"]
        self.token_to_idx = {t: i for i, t in enumerate(self.tokens)}
        self.idx_to_token = {i: t for t, i in self.token_to_idx.items()}
        self.vocab_size = len(self.tokens)

    def arity(self, token: str) -> int:
        if token in self.OPERATORS:
            return self.OPERATORS[token][0]
        return 0

    def is_field(self, token: str) -> bool:
        return token in self.fields

    def is_constant(self, token: str) -> bool:
        return token in self.constants

    def is_special(self, token: str) -> bool:
        return token in ("BEG", "SEP")

    def is_minute_field(self, token: str) -> bool:
        return token in self.RAW_MINUTE_FIELDS

    def valid_action_mask(self, stack_depth: int, step: int, max_len: int) -> List[int]:
        """
        构造下一步可行动作掩码。
        用于 MaskablePPO：1 表示可选，0 表示不可选。
        """
        mask = [0] * self.vocab_size
        for token, idx in self.token_to_idx.items():
            ok = False
            if token == "BEG":
                ok = False
            elif token == "SEP":
                # 只允许栈深恰好为 1 时收尾：栈里剩多个值的表达式必然非法
                # （曾导致 476/507 个随机候选死于 '最终栈深度 N != 1'）。
                ok = stack_depth == 1 and step >= 2
            elif self.is_field(token) or self.is_constant(token):
                # 压栈后栈深 +1，必须给后面留够收拢步数：
                # 每多 1 层深度需要 1 个二元算子收拢，外加最后 1 步 SEP。
                ok = max_len - 1 - step >= stack_depth + 1
            else:
                arity = self.arity(token)
                if arity == 1:
                    # 一元算子不改栈深，同样要留够收拢步数
                    ok = stack_depth >= 1 and max_len - 1 - step >= stack_depth
                elif arity == 2:
                    # 二元算子收拢 1 层，是收尾阶段唯一该放行的算子
                    ok = stack_depth >= 2 and max_len - 1 - step >= stack_depth - 1
                elif arity == 3:
                    ok = stack_depth >= 3 and max_len - 1 - step >= stack_depth - 2
            mask[idx] = 1 if ok else 0

        if step >= max_len - 1:
            mask = [0] * self.vocab_size
            mask[self.token_to_idx["SEP"]] = 1
        return mask

    def sample_expression(self, rng, max_len: int = 12) -> List[str]:
        """随机采样一个合法 RPN 表达式（用于 baseline / 测试）。"""
        expr = ["BEG"]
        stack_depth = 0
        for step in range(1, max_len):
            mask = self.valid_action_mask(stack_depth, step, max_len)
            valid_idx = [i for i, m in enumerate(mask) if m]
            if not valid_idx:
                break
            idx = rng.choice(valid_idx)
            token = self.idx_to_token[idx]
            expr.append(token)
            arity = self.arity(token)
            if arity == 0 and not self.is_special(token):
                stack_depth += 1
            else:
                stack_depth += 1 - arity
            if token == "SEP":
                break
        return expr


# 全局默认词表
VOCAB = Vocabulary()

# ================= RPN Evaluator (src/rpn_eval.py) =================
"""
RPN 向量化求值器（支持日频面板 + 分钟面板）

输入 DataFrame 约定：
- 日频：列包含 date, instrument，index 可以是默认整数或 (date, instrument)
- 分钟：列包含 date, time, instrument，index 可以是默认整数或 (date, time, instrument)

输出：
- 如果表达式最终是分钟序列，自动取 Day_Last 转成日频 (date, instrument) 序列。
- 如果表达式非法，返回 None。
"""

import re
import numpy as np
import pandas as pd
from typing import List, Optional, Union


class InvalidExpressionError(Exception):
    pass


Number = Union[int, float]
StackItem = Union[Number, pd.Series]


def _to_series_index(df: pd.DataFrame) -> pd.DataFrame:
    """把 date/time/instrument 列转成合适的 MultiIndex；已索引则直接返回，避免重复复制排序。"""
    idx_names = list(df.index.names)
    if idx_names[:2] == ["date", "instrument"] and "time" not in df.columns:
        return df
    if idx_names[:3] == ["date", "time", "instrument"]:
        return df
    if "time" in df.columns:
        # 分钟面板
        cols = ["date", "time", "instrument"]
        if not df.index.equals(pd.RangeIndex(len(df))):
            df = df.reset_index(drop=True)
        for c in cols:
            if c not in df.columns:
                raise ValueError(f"分钟面板缺少列 {c}")
        df = df.set_index(cols).sort_index()
    elif "date" in df.columns and "instrument" in df.columns:
        df = df.set_index(["date", "instrument"]).sort_index()
    return df


def _freq(item: StackItem) -> Optional[str]:
    if isinstance(item, pd.Series):
        if item.index.nlevels == 3:
            return "minute"
        elif item.index.nlevels == 2:
            return "daily"
    return None


def _broadcast_to_minute(daily_series: pd.Series, minute_index: pd.MultiIndex) -> pd.Series:
    """把日频序列广播到分钟索引。"""
    daily_series = daily_series.dropna()
    mapper = {(d, i): v for (d, i), v in daily_series.items()}
    out = []
    for d, t, i in minute_index:
        out.append(mapper.get((d, i), np.nan))
    return pd.Series(out, index=minute_index)


def _align_operands(a: StackItem, b: StackItem) -> tuple:
    """对齐两个操作数的频率，日频需要时会广播到分钟。"""
    fa, fb = _freq(a), _freq(b)
    if fa == "daily" and fb == "minute":
        a = _broadcast_to_minute(a, b.index)
    elif fa == "minute" and fb == "daily":
        b = _broadcast_to_minute(b, a.index)
    return a, b


def _apply_to_all(items: List[StackItem]) -> List[StackItem]:
    """对多个操作数统一对齐到最高频（分钟 > 日频 > 标量）。"""
    has_minute = any(_freq(x) == "minute" for x in items)
    if not has_minute:
        return items
    minute_index = None
    for x in items:
        if _freq(x) == "minute":
            minute_index = x.index
            break
    out = []
    for x in items:
        if _freq(x) == "daily":
            out.append(_broadcast_to_minute(x, minute_index))
        else:
            out.append(x)
    return out


def _rolling_last_rank(y: np.ndarray) -> float:
    """滚动窗口内最后一个取值的百分比排名（平均秩），与 pandas rank(pct=True) 末位值一致。

    配合 rolling.apply(..., raw=True) 使用，避免逐窗口构造 Series，快几十倍。
    """
    last = y[-1]
    if np.isnan(last):
        return np.nan
    valid = ~np.isnan(y)
    n = int(valid.sum())
    if n == 0:
        return np.nan
    v = y[valid]
    less = int((v < last).sum())
    eq = int((v == last).sum())
    return (less + (eq + 1) / 2) / n


class RPNEvaluator:
    def __init__(self, vocab):
        self.vocab = vocab

    def evaluate(
        self,
        tokens: List[str],
        df: pd.DataFrame,
        precomputed: Optional[Dict[Tuple[str, str], pd.Series]] = None,
    ) -> Optional[pd.Series]:
        """求值 RPN token 序列。

        precomputed: {(field_token, unary_op_token): 预计算好的序列}。命中时把
        "字段 + 一元算子" 当作叶子直接压栈，跳过重复的 rolling/groupby 计算。
        """
        df = _to_series_index(df)
        stack: List[StackItem] = []

        i = 0
        n = len(tokens)
        while i < n:
            tok = tokens[i]

            # 命中预计算表：字段 + 一元算子视为一个叶子
            if precomputed and self.vocab.is_field(tok) and i + 1 < n:
                key = (tok, tokens[i + 1])
                if key in precomputed:
                    stack.append(precomputed[key])
                    i += 2
                    continue

            if tok == "BEG" or tok == "SEP":
                pass

            elif self.vocab.is_field(tok):
                if tok not in df.columns:
                    raise InvalidExpressionError(f"字段 {tok} 不在数据中")
                stack.append(df[tok])

            elif self.vocab.is_constant(tok):
                stack.append(float(tok))

            elif self.vocab.arity(tok) == 1:
                if len(stack) < 1:
                    raise InvalidExpressionError(f"算子 {tok} 缺少操作数")
                x = stack.pop()
                res = self._apply_unary(tok, x)
                stack.append(res)

            elif self.vocab.arity(tok) == 2:
                if len(stack) < 2:
                    raise InvalidExpressionError(f"算子 {tok} 缺少操作数")
                b = stack.pop()
                a = stack.pop()
                res = self._apply_binary(tok, a, b)
                stack.append(res)

            elif self.vocab.arity(tok) == 3:
                if len(stack) < 3:
                    raise InvalidExpressionError(f"算子 {tok} 缺少操作数")
                c = stack.pop()
                b = stack.pop()
                a = stack.pop()
                res = self._apply_ternary(tok, a, b, c)
                stack.append(res)
            else:
                raise InvalidExpressionError(f"未知 token {tok}")

            i += 1

        if len(stack) != 1:
            raise InvalidExpressionError(f"最终栈深度 {len(stack)} != 1")

        final = stack[0]
        if isinstance(final, pd.Series):
            if final.index.nlevels == 3:
                # 分钟序列自动取日末值
                final = final.groupby(level=["date", "instrument"]).last()
            return final
        else:
            raise InvalidExpressionError("最终结果是标量，无效")

    def _apply_unary(self, op: str, x: StackItem) -> StackItem:
        # 一元算子
        if op == "ABS":
            return np.abs(x)
        if op == "SIGN":
            return np.sign(x)
        if op == "NEG":
            return -x
        if op == "LOG":
            # 对数处理：取绝对值并截断极小值
            if isinstance(x, pd.Series):
                return np.log(x.astype(float).abs().clip(lower=1e-8))
            return np.log(max(abs(x), 1e-8))

        # 截面算子
        if op == "Cs_Rank":
            return self._cs_rank(x)
        if op == "Cs_Zscore":
            return self._cs_zscore(x)

        # 日频时序算子
        m = re.match(r"Ts_(Mean|Std|Delay|Rank|Max|Min|Sum|Zscore)_(\d+)", op)
        if m:
            func, window = m.group(1), int(m.group(2))
            if _freq(x) != "daily":
                raise InvalidExpressionError(f"{op} 只能作用于日频序列")
            return self._ts_op(x, func, window)

        # 分钟频时序算子
        m = re.match(r"Ms_(Mean|Std|Delay|Rank|Max|Min|Sum)_(\d+)", op)
        if m:
            func, window = m.group(1), int(m.group(2))
            if _freq(x) != "minute":
                raise InvalidExpressionError(f"{op} 只能作用于分钟序列")
            return self._ms_op(x, func, window)

        # 分钟 -> 日聚合算子
        if op.startswith("Day_"):
            if _freq(x) != "minute":
                raise InvalidExpressionError(f"{op} 只能作用于分钟序列")
            agg = op.split("_")[1].lower()
            return getattr(x.groupby(level=["date", "instrument"]), agg)()

        raise InvalidExpressionError(f"未实现算子 {op}")

    def _apply_binary(self, op: str, a: StackItem, b: StackItem) -> StackItem:
        a, b = _align_operands(a, b)
        if op == "ADD":
            return a + b
        if op == "SUB":
            return a - b
        if op == "MUL":
            return a * b
        if op == "DIV":
            if isinstance(b, pd.Series):
                b_safe = b.replace(0, np.nan)
            else:
                b_safe = b if b != 0 else np.nan
            return a / b_safe
        if op == "MAX":
            return np.maximum(a, b)
        if op == "MIN":
            return np.minimum(a, b)
        raise InvalidExpressionError(f"未实现二元算子 {op}")

    def _apply_ternary(self, op: str, a: StackItem, b: StackItem, c: StackItem) -> StackItem:
        if op != "If_Else":
            raise InvalidExpressionError(f"未实现三元算子 {op}")
        items = _apply_to_all([a, b, c])
        a, b, c = items
        cond = a > 0 if not isinstance(a, pd.Series) else a > 0
        return b.where(cond, c)

    # ------------------------------------------------------------------
    # 截面算子
    # ------------------------------------------------------------------
    def _cs_rank(self, x: StackItem) -> StackItem:
        if not isinstance(x, pd.Series):
            raise InvalidExpressionError("Cs_Rank 只能作用于序列")
        if x.index.nlevels == 2:
            return x.groupby(level="date").rank(pct=True)
        else:
            return x.groupby(level=["date", "time"]).rank(pct=True)

    def _cs_zscore(self, x: StackItem) -> StackItem:
        if not isinstance(x, pd.Series):
            raise InvalidExpressionError("Cs_Zscore 只能作用于序列")
        x = x.replace([np.inf, -np.inf], np.nan)
        if x.index.nlevels == 2:
            grouper = x.groupby(level="date")
        else:
            grouper = x.groupby(level=["date", "time"])
        with np.errstate(invalid="ignore"):
            mean = grouper.transform("mean")
            std = grouper.transform("std").replace(0, np.nan)
            return (x - mean) / std

    # ------------------------------------------------------------------
    # 日频时序算子
    # ------------------------------------------------------------------
    def _ts_op(self, x: pd.Series, func: str, window: int) -> pd.Series:
        if func == "Delay":
            return x.groupby(level="instrument").shift(window)
        if func == "Zscore":
            mean = self._ts_op(x, "Mean", window)
            std = self._ts_op(x, "Std", window).replace(0, np.nan)
            with np.errstate(invalid="ignore"):
                return (x - mean) / std
        if func == "Rank":
            return x.groupby(level="instrument").transform(
                lambda s: s.rolling(window=window, min_periods=2).apply(
                    _rolling_last_rank, raw=True
                )
            )
        return x.groupby(level="instrument").transform(
            lambda s: getattr(s.rolling(window=window, min_periods=2), func.lower())()
        )

    # ------------------------------------------------------------------
    # 分钟频时序算子
    # ------------------------------------------------------------------
    def _ms_op(self, x: pd.Series, func: str, window: int) -> pd.Series:
        grouper = x.groupby(level=["date", "instrument"])
        if func == "Delay":
            return grouper.shift(window)
        if func == "Rank":
            return grouper.transform(
                lambda s: s.rolling(window=window, min_periods=2).apply(
                    lambda y: y.rank(pct=True).iloc[-1], raw=False
                )
            )
        return grouper.transform(
            lambda s: getattr(s.rolling(window=window, min_periods=2), func.lower())()
        )


def safe_evaluate(tokens: List[str], df: pd.DataFrame, vocab=None) -> Optional[pd.Series]:
    """安全求值，非法表达式返回 None。"""
    if vocab is None:
        vocab = VOCAB
    ev = RPNEvaluator(vocab)
    try:
        return ev.evaluate(tokens, df)
    except Exception:
        return None


# ================= Evaluator (src/evaluator.py) =================
"""
本地比赛评分器

模拟平台预处理：
1. 去极值 + 标准化
2. BARRA 风格剔除（用可用字段做代理）
3. 计算 Rank IC、ICIR、多空夏普、Stress ICIR
4. Elastic Net 滚动回归 -> ModelScore
"""

import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple


def winsorize_mad(s: pd.Series, n: float = 3.0) -> pd.Series:
    """截面 MAD 去极值；对空分组/全 NaN 返回 NaN。"""
    def _winsor(x):
        x = x.replace([np.inf, -np.inf], np.nan)
        if x.dropna().empty:
            return pd.Series(np.nan, index=x.index)
        with np.errstate(invalid="ignore"):
            median = x.median()
            mad = (x - median).abs().median()
            upper = median + n * 1.4826 * mad
            lower = median - n * 1.4826 * mad
            return x.clip(lower=lower, upper=upper)
    return s.groupby(level="date", group_keys=False).apply(_winsor)


def zscore(s: pd.Series) -> pd.Series:
    """截面 z-score；对空分组/全 NaN/标准差为 0 返回 NaN。"""
    def _z(x):
        x = x.replace([np.inf, -np.inf], np.nan)
        if x.dropna().empty:
            return pd.Series(np.nan, index=x.index)
        with np.errstate(invalid="ignore"):
            mean = x.mean()
            std = x.std()
            if std == 0 or pd.isna(std):
                return pd.Series(np.nan, index=x.index)
            return (x - mean) / std
    return s.groupby(level="date", group_keys=False).apply(_z)


def preprocess_factor(s: pd.Series) -> pd.Series:
    """平台式预处理：MAD 去极值 + z-score。"""
    s = winsorize_mad(s)
    s = zscore(s)
    return s


def neutralize(
    factor: pd.Series,
    styles: pd.DataFrame,
    style_cols: List[str],
) -> pd.Series:
    """
    对风格因子做截面回归取残差。
    factor: MultiIndex (date, instrument)
    styles: DataFrame with same index and style_cols
    """
    df = pd.concat([factor.rename("y"), styles[style_cols]], axis=1)

    def _resid(x):
        orig_index = x.index
        x = x.dropna(subset=["y"])
        if x.empty:
            return pd.Series(np.nan, index=orig_index)
        y = x["y"].values
        X = x[style_cols].fillna(0).values
        if np.linalg.matrix_rank(X) < X.shape[1]:
            return pd.Series(np.nan, index=orig_index)
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
        resid = y - X @ beta
        return pd.Series(resid, index=x.index).reindex(orig_index)

    resid = df.groupby(level="date", group_keys=False).apply(_resid)
    return resid.reindex(factor.index)


def compute_forward_returns(
    close: pd.Series,
    periods: int = 1,
) -> pd.Series:
    """计算下期收益，避免未来函数。close 为 0（停牌/异常）导致的 inf 一律置为 NaN，
    否则会污染多空组合分组均值（mean=inf, std=nan），夏普无法计算。"""
    ret = close.groupby(level="instrument").shift(-periods) / close - 1
    return ret.replace([np.inf, -np.inf], np.nan)


def compute_ic(
    factor: pd.Series,
    returns: pd.Series,
    method: str = "spearman",
) -> pd.Series:
    """计算每日 Rank IC 序列（向量化实现，避免逐日 groupby.apply + spearman）。"""
    df = pd.concat([factor.rename("factor"), returns.rename("return")], axis=1).dropna()
    g = df.groupby(level="date")
    if method == "spearman":
        df["factor"] = g["factor"].rank()
        df["return"] = g["return"].rank()
        g = df.groupby(level="date")
    n = g.size()
    mean_f = g["factor"].mean()
    mean_r = g["return"].mean()
    f_vals = df["factor"]
    r_vals = df["return"]
    cov = (f_vals * r_vals).groupby(level="date").mean() - mean_f * mean_r
    var_f = (f_vals ** 2).groupby(level="date").mean() - mean_f ** 2
    var_r = (r_vals ** 2).groupby(level="date").mean() - mean_r ** 2
    with np.errstate(invalid="ignore", divide="ignore"):
        ic = cov / np.sqrt(var_f * var_r)
    ic[(n < 5) | (var_f <= 0) | (var_r <= 0)] = np.nan
    return ic


def compute_ic_metrics(ic_series: pd.Series) -> Dict[str, float]:
    ic_series = ic_series.dropna()
    if len(ic_series) == 0:
        return {"ic_mean": np.nan, "ic_std": np.nan, "icir": np.nan, "ic_ratio": np.nan}
    ic_mean = ic_series.mean()
    ic_std = ic_series.std()
    return {
        "ic_mean": ic_mean,
        "ic_std": ic_std,
        "icir": ic_mean / ic_std if ic_std > 0 else np.nan,
        "ic_ratio": (ic_series > 0).mean(),
    }


def long_short_sharpe(
    factor: pd.Series,
    returns: pd.Series,
    n_quantiles: int = 5,
    cost: float = 0.0,
) -> float:
    """多空组合日收益序列的年化夏普。"""
    df = pd.concat([factor.rename("factor"), returns.rename("return")], axis=1)

    def _daily_pnl(x):
        x = x.replace([np.inf, -np.inf], np.nan).dropna()  # 防御 inf（因子或收益异常值）
        if len(x) < n_quantiles * 2:
            return np.nan
        try:
            x["q"] = pd.qcut(x["factor"], n_quantiles, labels=False, duplicates="drop")
        except Exception:
            return np.nan
        q_vals = set(x["q"].dropna().unique())
        if not (0 in q_vals and n_quantiles - 1 in q_vals):
            return np.nan
        long_ret = x[x["q"] == n_quantiles - 1]["return"].mean()
        short_ret = x[x["q"] == 0]["return"].mean()
        return long_ret - short_ret - 2 * cost

    pnl = df.groupby(level="date", group_keys=False).apply(_daily_pnl)
    pnl_valid = pnl.dropna()
    if len(pnl_valid) < 5:
        return np.nan
    if pnl_valid.std() == 0 or pd.isna(pnl_valid.std()):
        return np.nan
    return pnl_valid.mean() / pnl_valid.std() * np.sqrt(252)


def stress_icir(
    ic_series: pd.Series,
    stress_mask: Optional[pd.Series] = None,
) -> float:
    """特殊行情下的 ICIR。未提供 stress_mask 时默认取 IC 绝对值最高的 20% 交易日。"""
    ic_series = ic_series.dropna()
    if stress_mask is None:
        threshold = ic_series.abs().quantile(0.8)
        stress_mask = ic_series.abs() >= threshold
        ic_stress = ic_series[stress_mask]
    else:
        # 按归一化日期对齐 mask 再取布尔值，避免 ic_series dropna 后
        # 与 mask 长度/顺序不一致时布尔索引按位置错位（pandas 不报错但结果错）
        m = stress_mask.copy()
        m.index = pd.to_datetime(m.index).normalize()
        m = m.groupby(level=0).any()
        ic_dates = pd.to_datetime(ic_series.index).normalize()
        flags = np.array([bool(m.get(d, False)) for d in ic_dates])
        ic_stress = ic_series[flags]
    if len(ic_stress) < 3:
        return np.nan
    return ic_stress.mean() / ic_stress.std()


STRESS_PERIODS: List[Tuple[str, str]] = [
    ("2020-02-01", "2020-03-31"),   # 疫情极端冲击
    ("2022-03-01", "2022-04-30"),   # 地缘 + 疫情双重冲击
    ("2024-01-01", "2024-02-29"),   # 流动性负反馈危机
]


def make_stress_mask(dates: pd.DatetimeIndex, periods: List[Tuple[str, str]] = None) -> pd.Series:
    """根据给定日期区间生成压力测试掩码。dates 为日期索引（date 级别）。"""
    if periods is None:
        periods = STRESS_PERIODS
    dates = pd.to_datetime(dates).normalize()
    mask = pd.Series(False, index=dates)
    for start, end in periods:
        start_dt = pd.to_datetime(start).normalize()
        end_dt = pd.to_datetime(end).normalize()
        mask |= (dates >= start_dt) & (dates <= end_dt)
    return mask


def simulate_elastic_net_score(
    factors_df: pd.DataFrame,
    returns: pd.Series,
    train_window: int = 60,
    step: int = 20,
    alpha: float = 1e-3,
    l1_ratio: float = 0.9,
) -> Dict[str, float]:
    """
    模拟平台 B 项 Elastic Net 滚动回归。
    factors_df: index (date, instrument), columns 为候选因子
    returns: index (date, instrument), 值为下期标准化收益
    返回每个因子的 ModelScore。
    """
    from sklearn.linear_model import ElasticNet

    dates = factors_df.index.get_level_values("date").unique().sort_values()
    factor_cols = factors_df.columns.tolist()
    weights = {c: [] for c in factor_cols}

    for start_idx in range(0, len(dates) - train_window, step):
        train_dates = dates[start_idx: start_idx + train_window]
        test_dates = dates[start_idx + train_window: start_idx + train_window + step]
        if len(test_dates) == 0:
            break

        X_train = factors_df.loc[train_dates].fillna(0).values
        y_train = returns.loc[train_dates].fillna(0).values
        X_test = factors_df.loc[test_dates].fillna(0).values

        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000, fit_intercept=False)
        model.fit(X_train, y_train)
        coef = model.coef_

        # 用测试集 R2 或 IC 做简单校验；这里只记录系数
        for c, w in zip(factor_cols, coef):
            weights[c].append(abs(w))

    scores = {}
    for c in factor_cols:
        w_arr = np.array(weights[c])
        if len(w_arr) == 0 or w_arr.mean() < 1e-9:
            scores[c] = 0.0
        else:
            scores[c] = w_arr.mean() / (w_arr.std() + 1e-6)
    return scores


def competition_score(
    factor: pd.Series,
    returns: pd.Series,
    styles: Optional[pd.DataFrame] = None,
    style_cols: Optional[List[str]] = None,
    stress_mask: Optional[pd.Series] = None,
) -> Dict[str, float]:
    """
    计算单个因子在比赛规则下的 A 项代理得分。
    注意：A 项的 Rank 是绝对排名，这里只能算原始指标。
    """
    factor = preprocess_factor(factor)
    if styles is not None and style_cols:
        factor = neutralize(factor, styles, style_cols)
        factor = preprocess_factor(factor)

    ic_series = compute_ic(factor, returns)
    metrics = compute_ic_metrics(ic_series)
    sr = long_short_sharpe(factor, returns)
    stress = stress_icir(ic_series, stress_mask)

    # 这里返回的是绝对值，不是百分位。要得到 A 项需要在因子池间做 rank。
    return {
        "ic_mean": metrics["ic_mean"],
        "icir": metrics["icir"],
        "long_short_sharpe": sr,
        "stress_icir": stress,
    }


def prepare_panel(df: pd.DataFrame) -> pd.DataFrame:
    """把 DataFrame 转成 (date, instrument) 索引，方便后续计算。"""
    if "date" in df.columns and "instrument" in df.columns:
        df = df.set_index(["date", "instrument"]).sort_index()
    return df

# ================= Selector (src/selector.py) =================
"""
因子筛选层：相关去重 + LASSO / Elastic Net 选择
"""

import numpy as np
import pandas as pd
from typing import List


def correlation_filter(
    factors_df: pd.DataFrame,
    threshold: float = 0.95,
) -> List[str]:
    """
    截面相关性去重：保留 ICIR 更高的因子，移除与已保留因子相关性过高的因子。
    需要 factors_df 已经是 (date, instrument) 索引。
    """
    cols = factors_df.columns.tolist()
    if len(cols) <= 1:
        return cols

    # 计算每个因子的日度 rank 序列，再算因子间平均绝对相关
    rank_df = factors_df.groupby(level="date", group_keys=False).rank(pct=True)
    corr = rank_df.corr().abs()

    # 简单贪婪去重：按列顺序保留（调用前可先用 icir 排序）
    keep = []
    for c in cols:
        if not keep:
            keep.append(c)
            continue
        max_corr = corr.loc[c, keep].max()
        if max_corr < threshold:
            keep.append(c)
    return keep


def lasso_select(
    factors_df: pd.DataFrame,
    returns: pd.Series,
    alpha: float = 1e-4,
    l1_ratio: float = 0.9,
    train_dates: List = None,
) -> List[str]:
    """
    用 ElasticNet 选择非零系数因子。
    目标：下期标准化收益 y。
    """
    from sklearn.linear_model import ElasticNet

    if train_dates is not None:
        X = factors_df.loc[train_dates]
        y = returns.loc[train_dates]
    else:
        X = factors_df
        y = returns

    # 处理 inf / NaN，避免 sklearn 报错
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    y = y.replace([np.inf, -np.inf], np.nan).fillna(0)

    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000, fit_intercept=False)
    model.fit(X.values, y.values)

    selected = [c for c, w in zip(X.columns, model.coef_) if abs(w) > 1e-8]
    return selected


def ic_threshold_select(
    factors_df: pd.DataFrame,
    returns: pd.Series,
    min_abs_ic: float = 0.01,
    min_icir: float = 0.3,
) -> List[str]:
    """先按 IC/ICIR 阈值粗筛。"""

    selected = []
    for c in factors_df.columns:
        ic = compute_ic(factors_df[c], returns)
        metrics = compute_ic_metrics(ic)
        if abs(metrics["ic_mean"]) >= min_abs_ic and abs(metrics["icir"]) >= min_icir:
            selected.append(c)
    return selected


def _combine_factors(factors_df: pd.DataFrame, returns: pd.Series) -> pd.Series:
    """
    把多个因子等权合成：每个因子先做截面排名，再用训练集 IC 方向对齐符号。
    """

    signed_ranks = []
    for c in factors_df.columns:
        ic = compute_ic(factors_df[c], returns)
        sign = -1 if ic.mean() < 0 else 1
        rank = factors_df[c].groupby(level="date", group_keys=False).rank(pct=True)
        signed_ranks.append(rank * sign)
    combined = pd.concat(signed_ranks, axis=1).mean(axis=1)
    return combined


def incremental_pool_select(
    factors_df: pd.DataFrame,
    returns: pd.Series,
    train_dates: List = None,
    corr_threshold: float = 0.8,
    max_factors: int = 10,
    min_improvement: float = 0.01,
) -> List[str]:
    """
    贪婪增量入池：每次加入能带来最大 ICIR 提升、且与已选因子相关性不超过阈值的因子。

    这比 LASSO 更适合"相关性不高于 X% + 只有增量 ICIR 才入池"的需求。
    """

    if train_dates is None:
        train_dates = factors_df.index.get_level_values("date").unique()

    X_train = factors_df.loc[train_dates]
    y_train = returns.loc[train_dates]

    # 预计算每个候选在训练集上的 |ICIR|
    candidate_scores = {}
    for c in X_train.columns:
        ic = compute_ic(X_train[c], y_train)
        metrics = compute_ic_metrics(ic)
        candidate_scores[c] = abs(metrics.get("icir", 0))

    # 按 ICIR 从高到低尝试
    candidates = sorted(X_train.columns, key=lambda c: candidate_scores[c], reverse=True)
    selected = []
    best_icir = -np.inf

    while candidates and len(selected) < max_factors:
        best_candidate = None
        best_combined_icir = best_icir

        for c in candidates:
            # 相关性约束：与已选因子的最大绝对相关
            if selected:
                corrs = []
                for s in selected:
                    corr = X_train[c].corr(X_train[s])
                    if not pd.isna(corr):
                        corrs.append(abs(corr))
                if corrs and max(corrs) > corr_threshold:
                    continue

            trial_pool = selected + [c]
            combined = _combine_factors(X_train[trial_pool], y_train)
            ic = compute_ic(combined, y_train)
            metrics = compute_ic_metrics(ic)
            icir = abs(metrics.get("icir", 0))

            if icir > best_combined_icir:
                best_combined_icir = icir
                best_candidate = c

        # 没有满足约束且能提升的因子
        if best_candidate is None:
            break

        # 增量 ICIR 必须超过门槛
        improvement = best_combined_icir - best_icir
        if best_icir != -np.inf and improvement < min_improvement:
            break

        selected.append(best_candidate)
        candidates.remove(best_candidate)
        best_icir = best_combined_icir

    return selected


def select_factors(
    factors_df: pd.DataFrame,
    returns: pd.Series,
    train_dates: List = None,
    corr_threshold: float = 0.95,
    lasso_alpha: float = 1e-4,
    method: str = "incremental",
) -> List[str]:
    """
    完整筛选流程：
    1. IC 阈值粗筛
    2. 相关性去重
    3. LASSO 精选 或 增量 ICIR 入池

    method:
        - "lasso": ElasticNet 非零系数法（默认）
        - "incremental": 贪婪增量入池，满足 corr_threshold 且每次提升 ICIR
    """
    # 1. 粗筛
    coarse = ic_threshold_select(factors_df, returns)
    if not coarse:
        return []
    df_coarse = factors_df[coarse]

    # 2. 相关性去重（按 ICIR 降序）
    icir_scores = {}
    for c in coarse:
        metrics = compute_ic_metrics(compute_ic(df_coarse[c], returns))
        icir_scores[c] = abs(metrics.get("icir", 0))
    ordered = sorted(coarse, key=lambda x: icir_scores[x], reverse=True)
    dedup = correlation_filter(df_coarse[ordered], threshold=corr_threshold)

    if method == "incremental":
        selected = incremental_pool_select(
            factors_df[dedup],
            returns,
            train_dates=train_dates,
            corr_threshold=corr_threshold,
        )
    else:
        selected = lasso_select(factors_df[dedup], returns, alpha=lasso_alpha, train_dates=train_dates)
    return selected

# ================= Synthesizer (src/synthesizer.py) =================
"""
因子合成层：IC 加权 / XGBoost / 等权组合
"""

import numpy as np
import pandas as pd
from typing import Dict, List


def ic_weighted_combine(
    factors_df: pd.DataFrame,
    returns: pd.Series,
    train_dates: List = None,
    min_weight: float = 0.0,
) -> pd.Series:
    """
    按训练期内 ICIR 加权合成一个因子。
    权重 = ICIR / sum(abs(ICIR))，再做符号对齐（IC 为负的因子反向）。
    """

    if train_dates is not None:
        train_df = factors_df.loc[train_dates]
        train_ret = returns.loc[train_dates]
    else:
        train_df = factors_df
        train_ret = returns

    weights = {}
    signs = {}
    for c in factors_df.columns:
        ic = compute_ic(train_df[c], train_ret)
        metrics = compute_ic_metrics(ic)
        icir = metrics.get("icir", 0)
        ic_mean = metrics.get("ic_mean", 0)
        if pd.isna(icir):
            weights[c] = 0.0
            signs[c] = 1
        else:
            # 符号对齐：IC 为负的因子先反向，权重用 |ICIR|
            weights[c] = abs(icir)
            signs[c] = -1 if ic_mean < 0 else 1

    total = sum(abs(w) for w in weights.values())
    if total < 1e-9:
        return pd.Series(np.nan, index=factors_df.index)

    weights = {c: w / total for c, w in weights.items()}

    out = None
    for c, w in weights.items():
        contrib = factors_df[c] * signs[c] * w
        out = contrib if out is None else out + contrib
    return out


def xgb_combine(
    factors_df: pd.DataFrame,
    returns: pd.Series,
    train_dates: List,
    test_dates: List = None,
    params: Dict = None,
) -> pd.Series:
    """
    用 XGBoost 在训练期内拟合候选因子 -> 下期收益，在测试期预测。
    返回预测得分作为最终因子。
    """
    try:
        import xgboost as xgb
    except ImportError:
        raise ImportError("xgboost 未安装，请使用 ic_weighted_combine 或安装 xgboost")

    if params is None:
        params = {
            "objective": "reg:squarederror",
            "n_estimators": 200,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
            "random_state": 42,
        }

    X_train = factors_df.loc[train_dates].fillna(0)
    y_train = returns.loc[train_dates].fillna(0)

    model = xgb.XGBRegressor(**params)
    model.fit(X_train.values, y_train.values)

    if test_dates is not None:
        X_test = factors_df.loc[test_dates].fillna(0)
        pred = model.predict(X_test.values)
        return pd.Series(pred, index=X_test.index)

    X_all = factors_df.fillna(0)
    pred = model.predict(X_all.values)
    return pd.Series(pred, index=X_all.index)


def equal_weight_combine(factors_df: pd.DataFrame) -> pd.Series:
    """等权合成（baseline）。"""
    return factors_df.mean(axis=1)

# ================= LLM Generator (src/generator_llm.py) =================
"""
LLM 因子注入器（比赛版本）

由于比赛平台无外部网络，所有 LLM 调用必须：
1. 在线下完成，生成候选表达式列表保存到 JSON；或
2. 在平台使用本地部署模型（如 ollama/qwen）。

本模块提供：
- Prompt 模板
- 解析 LLM 输出为 RPN token 序列
- 候选表达式保存/加载
"""

import json
import os
import re
from typing import List, Optional


SYSTEM_PROMPT = """你是一名量化研究员，正在参加 A 股 AI 因子挖掘比赛。
比赛数据：中证1000 成分股，2019-2024 年，包含日频因子库和 1 分钟行情数据。

任务：基于高频量价、盘口、反转、波动等逻辑，生成 5 个有效的 RPN 因子表达式。

可用字段（部分示例）：
- 日频：close, volume, amount, turn, change_ratio, daily_return, momentum_5, reversal_5, volatility_5, total_market_cap, pe_ttm, pb, netflow_amount_main, beta_000300SH_22, roe_avg_ttm...
- 分钟聚合：realized_vol, intraday_return, morning_return, afternoon_return, vwap_dev, ob_imbalance_l5, spread_l1_mean...
- 分钟原始：m_close, m_volume, m_amount, m_ask_price1, m_bid_price1, m_ask_volume1, m_bid_volume1...

可用算子：
- 一元：ABS, SIGN, NEG, LOG
- 截面：Cs_Rank, Cs_Zscore
- 日频时序：Ts_Mean_5/10/20, Ts_Std_5/20, Ts_Delay_1/5, Ts_Rank_5/20, Ts_Max_5, Ts_Min_5, Ts_Sum_5, Ts_Zscore_20
- 分钟时序：Ms_Mean_5/30, Ms_Std_5/30, Ms_Delay_1/5, Ms_Rank_5/30, Ms_Max_5, Ms_Min_5, Ms_Sum_5
- 分钟->日：Day_Last, Day_Mean, Day_Std, Day_Max, Day_Min, Day_Sum
- 二元：ADD, SUB, MUL, DIV, MAX, MIN
- 三元：If_Else

要求：
1. 每个表达式以 BEG 开头，SEP 结尾，使用逆波兰 notation（RPN）。
2. 每个表达式附带一句经济逻辑解释。
3. 尽量覆盖不同方向（positive/negative）和不同逻辑（量价/波动/反转/盘口/资金流）。

输出格式（严格 JSON 数组）：
[
  {"expr": ["BEG", "close", "close", "Ts_Mean_5", "SUB", "close", "DIV", "SEP"], "logic": "收盘价相对5日均价的偏离，衡量短期反转"},
  ...
]
"""


# 方案 C（仅用平台日频因子库）专用的 prompt，避免 LLM 生成分钟级字段导致无法求值
DAILY_PROMPT = """你是一名量化研究员，正在参加 A 股 AI 因子挖掘比赛。

本次比赛仅使用平台日频因子库 `bigalpha_2026_factorlib`，不允许提交外部数据。
请基于日频量价、技术、资金流、市值等字段，生成 {n} 个有效的 RPN 因子表达式。

可用字段（严格只能使用以下字段名）：
- 价格/成交：close, volume, amount, turn, change_ratio, daily_return
- 技术因子：momentum_5, reversal_5, volatility_5, sma_20, ema_20, macd_diff_12_26_9, macd_dea_12_26_9, macd_hist_12_26_9, rsi_12, kdj_k_9_3_3, kdj_d_9_3_3, bias_20, cci_14, atr_14
- 估值/市值：total_market_cap, float_market_cap, pe_ttm, pb, ps_ttm, list_days
- 资金/风险：netflow_amount_main, netflow_amount_rate_main, net_active_buy_amount_main, beta_000300SH_22

可用算子：
- 一元：ABS, SIGN, NEG, LOG
- 二元：ADD, SUB, MUL, DIV, MAX, MIN
- 截面：Cs_Rank, Cs_Zscore
- 时序：Ts_Mean_5/10/20, Ts_Std_5/20, Ts_Delay_1/5, Ts_Rank_5/20, Ts_Max_5, Ts_Min_5, Ts_Sum_5, Ts_Zscore_20
- 三元：If_Else

要求：
1. 每个表达式必须以 BEG 开头、SEP 结尾，使用逆波兰 notation（RPN）。
2. 操作数数量必须和算子 arity 匹配。特别注意：二元算子（SUB/DIV/ADD/MUL/MAX/MIN）需要两个操作数。
   - 示例 1：(close - Ts_Mean_5(close)) / close  =>  ["BEG", "close", "close", "Ts_Mean_5", "SUB", "close", "DIV", "SEP"]
   - 示例 2：-(rsi_12 - Ts_Mean_10(rsi_12))    =>  ["BEG", "rsi_12", "rsi_12", "Ts_Mean_10", "SUB", "NEG", "SEP"]
   - 示例 3：Cs_Rank(turn) - Cs_Rank(total_market_cap) => ["BEG", "turn", "Cs_Zscore", "total_market_cap", "Cs_Zscore", "SUB", "SEP"]
3. 只能使用上面列出的字段名，不要使用任何分钟级字段（如 realized_vol、morning_return、m_close 等）。
4. 每个表达式附带一句经济逻辑解释。
5. 尽量覆盖不同方向（positive/negative）和不同逻辑（量价/波动/反转/资金流/估值）。

输出格式（严格 JSON 数组）：
[
  {"expr": ["BEG", "close", "close", "Ts_Mean_5", "SUB", "close", "DIV", "SEP"], "logic": "收盘价相对5日均价的偏离，衡量短期反转"},
  {"expr": ["BEG", "rsi_12", "rsi_12", "Ts_Mean_10", "SUB", "NEG", "SEP"], "logic": "RSI相对10日均值的下穿幅度"},
  ...
]
"""


def parse_llm_output(text: str) -> List[List[str]]:
    """从 LLM 输出中提取 RPN 表达式列表。"""
    expressions = []
    # 尝试提取 JSON
    try:
        data = json.loads(text)
        for item in data:
            if isinstance(item, dict) and "expr" in item:
                expressions.append(item["expr"])
            elif isinstance(item, list):
                expressions.append(item)
    except Exception:
        # 退化为正则提取方括号数组
        for match in re.findall(r'\[[^\]]+\]', text):
            try:
                expr = json.loads(match.replace("'", '"'))
                if isinstance(expr, list) and expr[0] == "BEG":
                    expressions.append(expr)
            except Exception:
                continue
    return expressions


def save_expressions(expressions: List[List[str]], path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(expressions, f, ensure_ascii=False, indent=2)


def load_expressions(path: str) -> List[List[str]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _format_prompt(prompt: str, n: int) -> str:
    """如果 prompt 包含 {n} 占位符，则注入生成数量。

    使用 replace 而非 str.format，避免 prompt 中 JSON 大括号被误解析为占位符。
    """
    return prompt.replace("{n}", str(n))


def _is_valid_rpn_stack(expr: List[str]) -> bool:
    """用栈模拟检查 RPN 表达式是否合法（操作数/算子数量匹配）。"""
    arities = {
        "ABS": 1, "SIGN": 1, "NEG": 1, "LOG": 1,
        "Cs_Rank": 1, "Cs_Zscore": 1,
        "ADD": 2, "SUB": 2, "MUL": 2, "DIV": 2, "MAX": 2, "MIN": 2,
        "If_Else": 3,
    }
    import re
    for tok in expr:
        if re.match(r"Ts_.*?_\d+", tok) or re.match(r"Ms_.*?_\d+", tok):
            arities[tok] = 1

    stack = 0
    for tok in expr:
        if tok in ("BEG", "SEP"):
            continue
        if tok in arities:
            arity = arities[tok]
            if stack < arity:
                return False
            stack = stack - arity + 1
        else:
            stack += 1
    return stack == 1


def filter_valid_expressions(expressions: List[List[str]]) -> List[List[str]]:
    """过滤掉包含词表外 token 或栈不合法的表达式。"""
    try:
        valid_tokens = set(VOCAB.tokens)
    except Exception:
        return expressions

    def _valid(expr):
        tokens_ok = all(str(tok) in valid_tokens for tok in expr)
        return tokens_ok and _is_valid_rpn_stack(expr)

    return [expr for expr in expressions if _valid(expr)]


def offline_llm_generate(
    prompt: str = DAILY_PROMPT,
    api_key: Optional[str] = None,
    base_url: Optional[str] = None,
    model: str = "glm-chat",
    n: int = 5,
    temperature: float = 0.8,
    json_mode: bool = True,
) -> List[List[str]]:
    api_key = api_key or os.environ.get("LLM_API_KEY")
    base_url = base_url or os.environ.get("LLM_BASE_URL")
    model = os.environ.get("LLM_MODEL", model)
    env_json_mode = os.environ.get("LLM_JSON_MODE", "1" if json_mode else "0")
    json_mode = env_json_mode.lower() in ("1", "true", "yes")
    timeout = int(os.environ.get("LLM_TIMEOUT", "300"))

    if not api_key:
        raise ValueError("请提供 api_key 或设置环境变量 LLM_API_KEY")

    prompt = _format_prompt(prompt, n)

    # 把生成数量注入 prompt，避免模型只返回固定个数
    user_prompt = f"请生成 {n} 个有效的 RPN 因子表达式，严格按下面 JSON 数组格式输出，只输出 JSON，不要额外解释。"

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
    }
    if json_mode:
        payload["response_format"] = {"type": "json_object"}

    # 使用 urllib 直接调用 OpenAI 兼容接口，避免 openai/pydantic 依赖冲突
    import urllib.request
    import urllib.error

    url = base_url.rstrip("/") + "/chat/completions"
    req = urllib.request.Request(
        url,
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}",
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            response_data = json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="ignore")
        raise RuntimeError(f"LLM API 请求失败: {e.code} {body}") from e

    content = response_data["choices"][0]["message"]["content"]

    # 先尝试直接解析 JSON；失败则用 parse_llm_output 做容错解析（兼容 markdown / 多段输出）
    try:
        data = json.loads(content)
    except Exception:
        expressions = parse_llm_output(content)
        return filter_valid_expressions(expressions)

    # 兼容 LLM 直接返回数组或包在 {"expressions": [...]} 里的情况
    if isinstance(data, dict):
        for key in ("expressions", "factors", "result", "data"):
            if key in data and isinstance(data[key], list):
                data = data[key]
                break
        else:
            data = []

    expressions = []
    for item in data:
        if isinstance(item, dict) and "expr" in item:
            expr = item["expr"]
        elif isinstance(item, list):
            expr = item
        else:
            continue
        if isinstance(expr, list) and len(expr) >= 2 and expr[0] == "BEG":
            expressions.append(expr)

    return filter_valid_expressions(expressions)

# ================= Data Loader (src/data_loader.py) =================
"""
数据加载层

- 日频因子库：bigalpha_2026_factorlib
- 成分股表：bigalpha_2026_instruments
- 1 分钟行情库：物理表名由 main 的 datasources["bar1m"] 注入（公榜默认 bigalpha_2026_stock_bar1m）

所有函数都返回 pandas DataFrame，index 不做强制要求，下游会自己处理。
"""

import time
import numpy as np
import pandas as pd
from typing import List, Optional


def _to_date_str(x) -> str:
    return pd.to_datetime(x).strftime("%Y-%m-%d")


def get_csi1000_instruments(dates: List[str]) -> pd.DataFrame:
    """获取指定日期区间内的中证1000成分股。"""
    import dai
    start, end = min(dates), max(dates)
    sql = f"""
        SELECT date, instrument
        FROM bigalpha_2026_instruments
        WHERE date >= '{start}' AND date <= '{end}'
    """
    df = dai.query(sql, filters={"date": [start, end]}).df()
    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    return df


def _date_chunks(start_date: str, end_date: str, freq: str = "YS") -> List[Tuple[str, str]]:
    """把日期区间拆成不重叠的小段，避免单次查询过大返回空。"""
    dates = pd.date_range(start=start_date, end=end_date, freq=freq)
    if len(dates) == 0:
        return [(start_date, end_date)]
    boundaries = [pd.to_datetime(start_date)] + dates.tolist()[1:] + [pd.to_datetime(end_date)]
    chunks = []
    for i in range(len(boundaries) - 1):
        s = boundaries[i].strftime("%Y-%m-%d")
        e = boundaries[i + 1].strftime("%Y-%m-%d")
        chunks.append((s, e))
    return chunks


def load_factor_library(
    start_date: str,
    end_date: str,
    instruments: Optional[List[str]] = None,
    fields: Optional[List[str]] = None,
    constituents_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """
    读取平台日频因子库。
    如果 instruments 为 None，则读取全市场（按 filters date 限制）。

    如果传入 constituents_df（含 date, instrument 两列），会按日期严格过滤，
    确保每天只保留当日成分股，避免用到非成分股或上市前数据。
    """
    import dai

    fields = fields or ["*"]
    if fields == ["*"]:
        select_clause = "*"
    else:
        select_clause = "date, instrument, " + ", ".join(fields)

    # 如果传了每日成分股表，按季分组查询：每个季度只用当季成分股 IN 列表，
    # 避免大区间全量查询或 IN 列表过长导致返回空。
    # （原为按月 72 次查询，改按季 24 次，减少 dai.query 往返开销；之后仍按日严格过滤，结果不变）
    if constituents_df is not None and not constituents_df.empty:
        parts = []
        inst_months = constituents_df.copy()
        inst_months["date"] = pd.to_datetime(inst_months["date"])
        inst_months["ym"] = inst_months["date"].dt.to_period("Q")
        start_dt, end_dt = pd.to_datetime(start_date), pd.to_datetime(end_date)
        months = pd.period_range(start=start_dt, end=end_dt, freq="Q")
        for per in months:
            sub = inst_months[inst_months["ym"] == per]
            if sub.empty:
                continue
            month_insts = sub["instrument"].unique().tolist()
            month_start = per.start_time.strftime("%Y-%m-%d")
            month_end = per.end_time.strftime("%Y-%m-%d")
            inst_str = ", ".join([f"'{x}'" for x in month_insts])
            month_sql = f"""
                SELECT {select_clause}
                FROM bigalpha_2026_factorlib
                WHERE instrument IN ({inst_str})
            """
            part = dai.query(month_sql, filters={"date": [month_start, month_end]}).df()
            print(f"  DEBUG {per}: insts={len(month_insts)}, part_shape={part.shape}")
            if not part.empty:
                parts.append(part)
        if not parts:
            print("DEBUG: 所有按月查询都为空")
            return pd.DataFrame(columns=["date", "instrument"] + (fields if fields != ["*"] else []))
        df = pd.concat(parts, ignore_index=True)
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
        print(f"DEBUG: concat 后 shape={df.shape}")
        # 再按日期严格过滤，确保每天只有当日成分股
        df = df.merge(
            constituents_df[["date", "instrument"]].drop_duplicates(),
            on=["date", "instrument"],
            how="inner",
        )
        print(f"DEBUG: merge 后 shape={df.shape}")
    elif instruments is not None and len(instruments) > 0:
        # 中证1000 只有 1000 只，IN 列表不会太长
        inst_str = ", ".join([f"'{x}'" for x in instruments])
        sql = f"""
            SELECT {select_clause}
            FROM bigalpha_2026_factorlib
            WHERE instrument IN ({inst_str})
        """
        df = dai.query(sql, filters={"date": [start_date, end_date]}).df()
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    else:
        sql = f"""
            SELECT {select_clause}
            FROM bigalpha_2026_factorlib
        """
        df = dai.query(sql, filters={"date": [start_date, end_date]}).df()
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")

    # 上市天数过滤先关闭：当前因子库中 list_days 列可能存在异常值（全 0/全 NaN），
    # 会误删全部数据。如需剔除新股，可在筛选/打分阶段另行处理。
    return df


def _query_minute_bars_day(
    date_str: str,
    instruments: List[str],
    fields: List[str],
    chunk_size: int = 100,
    bar1m_table: str = "bigalpha_2026_stock_bar1m",
) -> pd.DataFrame:
    """查询某一天的分钟行情数据，按股票分批。"""
    import dai

    # 平台 date 是 timestamp，用 filters 给出当天范围即可
    next_day = (pd.to_datetime(date_str) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    chunks = [
        instruments[i: i + chunk_size]
        for i in range(0, len(instruments), chunk_size)
    ]

    parts = []
    for chunk in chunks:
        inst_str = ", ".join([f"'{x}'" for x in chunk])
        sql = f"""
            SELECT date, instrument, {', '.join(fields)}
            FROM {bar1m_table}
            WHERE instrument IN ({inst_str})
        """
        df = dai.query(sql, filters={"date": [date_str, next_day]}).df()
        parts.append(df)

    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)


def _query_minute_bars_range(
    start_date: str,
    end_date: str,
    instruments: List[str],
    fields: List[str],
    chunk_size: int = 500,
    max_workers: int = 4,
    bar1m_table: str = "bigalpha_2026_stock_bar1m",
) -> pd.DataFrame:
    """查询一段日期区间内的分钟行情数据，按股票分批。

    分批查询用线程池并发执行：dai.query 的耗时主要是网络等待，线程并发即可
    显著缩短 wall time。若平台客户端并发调用报错，把 max_workers 设为 1 恢复串行。
    """
    import dai

    # 包含 end_date 全天
    next_day = (pd.to_datetime(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    chunks = [
        instruments[i: i + chunk_size]
        for i in range(0, len(instruments), chunk_size)
    ]

    def _fetch(chunk: List[str]) -> pd.DataFrame:
        inst_str = ", ".join([f"'{x}'" for x in chunk])
        sql = f"""
            SELECT date, instrument, {', '.join(fields)}
            FROM {bar1m_table}
            WHERE instrument IN ({inst_str})
        """
        return dai.query(sql, filters={"date": [start_date, next_day]}).df()

    if max_workers > 1 and len(chunks) > 1:
        with ThreadPoolExecutor(max_workers=min(max_workers, len(chunks))) as ex:
            parts = list(ex.map(_fetch, chunks))
    else:
        parts = [_fetch(chunk) for chunk in chunks]

    parts = [p for p in parts if p is not None and not p.empty]
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)


# 高频日频特征的下推聚合 SQL。方言经 experiments/SQL_CAPABILITY_PROBE.py 验证：
# DATE()/HOUR()/LAG/ROW_NUMBER/CASE WHEN in window ORDER BY/SUM(POW())/NULLIF 均可用。
# 注意口径修正（相对旧 pandas 路径）：
# - realized_vol / first / last：显式按时间排序（分钟数据返回行序实测是乱的）
# - vwap = sum(amount)/sum(volume)：volume/amount 是单分钟值而非日内累计（探针 T0 实测）
_HF_DAILY_SQL = """
WITH bars AS (
    SELECT DATE(date) AS d, instrument, date, open, close, volume, amount,
           ask_price1, bid_price1,
           ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5 AS ask_vol_sum,
           bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5 AS bid_vol_sum,
           bid_num_orders1, ask_num_orders1,
           LAG(close) OVER (PARTITION BY DATE(date), instrument ORDER BY date) AS prev_close,
           ROW_NUMBER() OVER (PARTITION BY DATE(date), instrument ORDER BY date ASC) AS rn_first,
           ROW_NUMBER() OVER (PARTITION BY DATE(date), instrument ORDER BY date DESC) AS rn_last,
           ROW_NUMBER() OVER (PARTITION BY DATE(date), instrument
               ORDER BY CASE WHEN HOUR(date) * 100 + MINUTE(date) <= 1030 THEN 1 ELSE 2 END, date ASC) AS rn_m_open,
           ROW_NUMBER() OVER (PARTITION BY DATE(date), instrument
               ORDER BY CASE WHEN HOUR(date) * 100 + MINUTE(date) <= 1030 THEN 1 ELSE 2 END, date DESC) AS rn_m_close,
           ROW_NUMBER() OVER (PARTITION BY DATE(date), instrument
               ORDER BY CASE WHEN HOUR(date) * 100 + MINUTE(date) >= 1400 THEN 1 ELSE 2 END, date ASC) AS rn_a_open,
           ROW_NUMBER() OVER (PARTITION BY DATE(date), instrument
               ORDER BY CASE WHEN HOUR(date) * 100 + MINUTE(date) >= 1400 THEN 1 ELSE 2 END, date DESC) AS rn_a_close
    FROM {bar1m_table}
    WHERE instrument IN ({inst_str})
)
SELECT d AS date, instrument,
    SQRT(SUM(POW(close / prev_close - 1, 2))) AS realized_vol,
    MAX(CASE WHEN rn_last = 1 THEN close END) / MAX(CASE WHEN rn_first = 1 THEN open END) - 1 AS intraday_return,
    MAX(CASE WHEN rn_m_close = 1 THEN close END) / MAX(CASE WHEN rn_m_open = 1 THEN open END) - 1 AS morning_return,
    MAX(CASE WHEN rn_a_close = 1 THEN close END) / MAX(CASE WHEN rn_a_open = 1 THEN close END) - 1 AS afternoon_return,
    MAX(CASE WHEN rn_last = 1 THEN close END) / (SUM(amount) / NULLIF(SUM(volume), 0)) - 1 AS vwap_dev,
    AVG((bid_vol_sum - ask_vol_sum) / NULLIF(bid_vol_sum + ask_vol_sum, 0)) AS ob_imbalance_l5,
    AVG(ask_price1 / NULLIF(bid_price1, 0) - 1) AS spread_l1_mean,
    AVG(bid_num_orders1) AS quote_intensity_bid,
    AVG(ask_num_orders1) AS quote_intensity_ask
FROM bars
GROUP BY d, instrument
"""


def _query_hf_daily_features_sql(
    start_date: str,
    end_date: str,
    instruments: List[str],
    chunk_size: int = 1000,
    max_workers: int = 4,
    bar1m_table: str = "bigalpha_2026_stock_bar1m",
) -> pd.DataFrame:
    """在平台侧把 1 分钟数据直接聚合成日频高频特征，只回传日频结果。

    传输量从分钟级（数百万行/月）降到日频级（数万行/季）。
    返回 DataFrame: date(str), instrument, [9 个 hf 特征]
    """
    import dai

    next_day = (pd.to_datetime(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    chunks = [instruments[i: i + chunk_size] for i in range(0, len(instruments), chunk_size)]

    def _fetch(chunk: List[str]) -> pd.DataFrame:
        inst_str = ", ".join(f"'{x}'" for x in chunk)
        return dai.query(_HF_DAILY_SQL.format(inst_str=inst_str, bar1m_table=bar1m_table),
                         filters={"date": [start_date, next_day]}).df()

    if max_workers > 1 and len(chunks) > 1:
        with ThreadPoolExecutor(max_workers=min(max_workers, len(chunks))) as ex:
            parts = list(ex.map(_fetch, chunks))
    else:
        parts = [_fetch(chunk) for chunk in chunks]

    parts = [p for p in parts if p is not None and not p.empty]
    if not parts:
        return pd.DataFrame()
    out = pd.concat(parts, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"]).dt.strftime("%Y-%m-%d")
    return out


def _aggregate_minute_bars(df_min: pd.DataFrame) -> pd.DataFrame:
    """把分钟数据聚合成日频高频特征。输入列：date, instrument, [分钟字段]。

    口径说明（探针 T0 实测）：
    - 分钟数据返回行序不保证按时间排列，pct_change / first / last 都依赖时序，
      必须先按 (instrument, 时间) 排序，否则 realized_vol 等全部算错；
    - volume/amount 是单分钟值而非日内累计，vwap = sum(amount)/sum(volume)
      （旧口径 last/last 使 vwap_dev 恒为 0）。
    性能说明：全程约 3 趟分组；float32 转换使内存带宽减半（分钟聚合对精度不敏感）。
    """
    float_cols = [c for c in df_min.columns if c not in ("date", "instrument")]
    df_min[float_cols] = df_min[float_cols].astype(np.float32)

    df_min["datetime"] = pd.to_datetime(df_min["date"])
    df_min["day"] = df_min["datetime"].dt.normalize()
    df_min["time"] = df_min["datetime"].dt.hour * 100 + df_min["datetime"].dt.minute
    df_min = df_min.sort_values(["instrument", "datetime"]).reset_index(drop=True)
    grp = ["day", "instrument"]

    # 单分钟收益平方（组内首根置 NaN，等价于 groupby.pct_change 的口径）
    group_changed = (
        df_min["instrument"].ne(df_min["instrument"].shift())
        | df_min["day"].ne(df_min["day"].shift())
    )
    prev_close = df_min["close"].shift()
    prev_close[group_changed] = np.nan
    df_min["ret2"] = (df_min["close"] / prev_close - 1) ** 2

    # 盘口指标逐行算好，随主 agg 一趟求均值
    bid_cols = [f"bid_volume{i}" for i in range(1, 6)]
    ask_cols = [f"ask_volume{i}" for i in range(1, 6)]
    bid_vol_sum = df_min[bid_cols].sum(axis=1)
    ask_vol_sum = df_min[ask_cols].sum(axis=1)
    df_min["ob_imbalance_l5"] = (bid_vol_sum - ask_vol_sum) / (bid_vol_sum + ask_vol_sum).replace(0, np.nan)
    df_min["spread_l1"] = df_min["ask_price1"] / df_min["bid_price1"].replace(0, np.nan) - 1

    agg = df_min.groupby(grp, sort=False).agg(
        open_first=("open", "first"),
        close_last=("close", "last"),
        volume_sum=("volume", "sum"),
        amount_sum=("amount", "sum"),
        ret2_sum=("ret2", "sum"),
        ob_imbalance_l5=("ob_imbalance_l5", "mean"),
        spread_l1_mean=("spread_l1", "mean"),
        quote_intensity_bid=("bid_num_orders1", "mean"),
        quote_intensity_ask=("ask_num_orders1", "mean"),
    )

    # 已实现波动
    agg["realized_vol"] = np.sqrt(agg["ret2_sum"])

    # 日内收益
    agg["intraday_return"] = agg["close_last"] / agg["open_first"] - 1

    # 早盘收益 9:30 - 10:30（首根 open 即全日首根 open，只需子集最后一根 close）
    morning_close = df_min[df_min["time"] <= 1030].groupby(grp, sort=False)["close"].last()
    agg["morning_return"] = morning_close / agg["open_first"] - 1

    # 尾盘收益 14:00 - 收盘（末根 close 即全日末根 close，只需子集第一根 close）
    afternoon_open = df_min[df_min["time"] >= 1400].groupby(grp, sort=False)["close"].first()
    agg["afternoon_return"] = agg["close_last"] / afternoon_open - 1

    # VWAP 偏离
    agg["vwap"] = agg["amount_sum"] / agg["volume_sum"].replace(0, np.nan)
    agg["vwap_dev"] = agg["close_last"] / agg["vwap"] - 1

    agg = agg.reset_index()
    agg["date"] = agg["day"].dt.strftime("%Y-%m-%d")
    return agg.drop(columns=["day"])


def _hf_agg_one_chunk(
    date_start: str,
    date_end: str,
    instruments: List[str],
    minute_fields: List[str],
    bar1m_table: str = "bigalpha_2026_stock_bar1m",
) -> pd.DataFrame:
    """loky worker 入口：下载一个 (月块 × 股票子块) 的分钟数据并聚合成日频特征。

    worker 自下载自聚合：分钟数据不经过父进程，无任何大对象序列化开销。
    （平台探针已验证 dai 在 loky worker 进程内可正常查询。）
    """
    import dai

    warnings.filterwarnings("ignore", category=RuntimeWarning)
    next_day = (pd.to_datetime(date_end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    inst_str = ", ".join(f"'{x}'" for x in instruments)
    sql = f"""
        SELECT date, instrument, {', '.join(minute_fields)}
        FROM {bar1m_table}
        WHERE instrument IN ({inst_str})
    """
    df_min = dai.query(sql, filters={"date": [date_start, next_day]}).df()
    if df_min.empty:
        return pd.DataFrame()
    return _aggregate_minute_bars(df_min)


def build_hf_daily_features(
    start_date: str,
    end_date: str,
    instruments: Optional[List[str]] = None,
    minute_fields: Optional[List[str]] = None,
    chunk_size: int = 500,
    date_freq: str = "M",
    verbose: bool = True,
    query_workers: int = 4,
    prefetch: bool = True,
    use_sql_pushdown: bool = False,
    agg_workers: int = -1,
    bar1m_table: str = "bigalpha_2026_stock_bar1m",
) -> pd.DataFrame:
    """
    从 1 分钟行情库聚合出日频高频特征（按月份分块向量化计算，避免逐日查询）。
    返回 DataFrame: date, instrument, [hf features]

    query_workers: 每个月块内按股票分批查询的线程并发数（1 = 完全串行）。
    prefetch: 聚合当前月块时在后台预取下一月块（False = 不预取，省内存）。
    use_sql_pushdown: True 时在平台侧直接 SQL 聚合出日频特征。实测平台分钟数据
        下载极快（0.8s/月/500股）而窗口函数很慢（25.5s/季/500股），故默认 False，
        走本地向量化聚合；SQL 路径仅留作备用。
    agg_workers: 本地聚合的 loky 并行进程数（-1 = 全部核，1 = 串行回退）。
        并行模式下每个 worker 自下载自聚合，分钟数据不进父进程。
    bar1m_table: 分钟 K 线物理表名（由 main 从平台 datasources["bar1m"] 注入，
        公榜/私榜自动切换，SQL 中不得硬编码）。
    """
    import dai

    if minute_fields is None:
        # 只查询实际用到的字段（open/close/volume/amount + 一档价 + 五档量 + 一档笔数）。
        # 原默认 26 个字段中 high/low 和 2~5 档价格从未参与任何特征计算，
        # 去掉后分钟数据传输量减少约 40%，聚合结果完全不变。
        minute_fields = (
            ["open", "close", "volume", "amount", "ask_price1", "bid_price1"]
            + [f"ask_volume{i}" for i in range(1, 6)]
            + [f"bid_volume{i}" for i in range(1, 6)]
            + ["ask_num_orders1", "bid_num_orders1"]
        )

    if instruments is None:
        instruments = get_csi1000_instruments([start_date, end_date])["instrument"].unique().tolist()

    # 非重叠的月份分块，避免逐日查询
    months = pd.period_range(start=start_date, end=end_date, freq=date_freq)
    date_chunks = [(p.start_time.strftime("%Y-%m-%d"), p.end_time.strftime("%Y-%m-%d")) for p in months]
    if verbose:
        print(f"高频特征构造：{len(date_chunks)} 个时间块, 股票数 {len(instruments)}")

    # ---------------- SQL 下推模式 ----------------
    # 聚合在平台侧完成，结果只有日频行（每季度数万行），无需按月切分；按季度分块查询。
    if use_sql_pushdown:
        quarters = pd.period_range(start=start_date, end=end_date, freq="Q")
        q_chunks = [(p.start_time.strftime("%Y-%m-%d"), p.end_time.strftime("%Y-%m-%d")) for p in quarters]
        if verbose:
            print(f"SQL 下推模式：{len(q_chunks)} 个季度块, 股票数 {len(instruments)}")
        sql_list = []

        def _fetch_hf(cs: str, ce: str) -> pd.DataFrame:
            return _query_hf_daily_features_sql(
                cs, ce, instruments, max_workers=query_workers, bar1m_table=bar1m_table
            )

        hf_pool = ThreadPoolExecutor(max_workers=1) if (prefetch and q_chunks) else None
        pending_hf = hf_pool.submit(_fetch_hf, *q_chunks[0]) if hf_pool is not None else None
        for qi, (cs, ce) in enumerate(q_chunks):
            t0 = time.time()
            if hf_pool is not None:
                agg_sql = pending_hf.result()
                if qi + 1 < len(q_chunks):
                    pending_hf = hf_pool.submit(_fetch_hf, *q_chunks[qi + 1])
            else:
                agg_sql = _fetch_hf(cs, ce)
            if agg_sql.empty:
                if verbose:
                    print(f"{cs} ~ {ce}: 无数据")
                continue
            sql_list.append(agg_sql)
            if verbose:
                print(f"{cs} ~ {ce}: shape={agg_sql.shape}, 耗时 {time.time()-t0:.2f}s")
        if hf_pool is not None:
            hf_pool.shutdown()
        if not sql_list:
            return pd.DataFrame()
        return pd.concat(sql_list, ignore_index=True)

    # ---------------- 并行聚合模式（默认） ----------------
    # 每个 loky 任务负责一个 (月块 × 股票子块)：worker 内自行 dai.query 下载并聚合，
    # 分钟数据不经过父进程，无大对象序列化；平台实测 dai 在 worker 可用。
    if Parallel is not None and agg_workers != 1:
        inst_chunks = [instruments[i: i + chunk_size] for i in range(0, len(instruments), chunk_size)]
        tasks = [(cs, ce, chunk) for (cs, ce) in date_chunks for chunk in inst_chunks]
        if verbose:
            print(f"并行聚合模式：{len(date_chunks)} 个月块 × {len(inst_chunks)} 批 = {len(tasks)} 个任务")
        parts = Parallel(n_jobs=agg_workers, backend="loky", verbose=10, batch_size=1)(
            delayed(_hf_agg_one_chunk)(cs, ce, chunk, minute_fields, bar1m_table)
            for cs, ce, chunk in tqdm(tasks, desc="构建高频特征")
        )
        parts = [p for p in parts if p is not None and not p.empty]
        if not parts:
            return pd.DataFrame()
        return pd.concat(parts, ignore_index=True)

    # ---------------- 串行回退（agg_workers=1 或无 joblib） ----------------
    daily_list = []

    def _fetch_chunk(cs: str, ce: str) -> pd.DataFrame:
        return _query_minute_bars_range(
            cs, ce, instruments, minute_fields, chunk_size, max_workers=query_workers,
            bar1m_table=bar1m_table
        )

    # 预取下一个月块：让下一块的 dai.query（网络 I/O）与当前块的 pandas 聚合（CPU）
    # 重叠执行。峰值内存约为同时持有 2 个月块的分钟数据，内存紧张时设 prefetch=False。
    fetch_pool = ThreadPoolExecutor(max_workers=1) if (prefetch and date_chunks) else None
    pending = fetch_pool.submit(_fetch_chunk, *date_chunks[0]) if fetch_pool is not None else None

    for chunk_idx, (chunk_start, chunk_end) in enumerate(date_chunks):
        t0 = time.time()
        if fetch_pool is not None:
            df_min = pending.result()
            if chunk_idx + 1 < len(date_chunks):
                pending = fetch_pool.submit(_fetch_chunk, *date_chunks[chunk_idx + 1])
        else:
            df_min = _fetch_chunk(chunk_start, chunk_end)
        if df_min.empty:
            if verbose:
                print(f"{chunk_start} ~ {chunk_end}: 无分钟数据")
            continue

        agg = _aggregate_minute_bars(df_min)
        daily_list.append(agg)

        if verbose:
            print(f"{chunk_start} ~ {chunk_end}: shape={agg.shape}, 耗时 {time.time()-t0:.2f}s")

    if fetch_pool is not None:
        fetch_pool.shutdown()

    if not daily_list:
        return pd.DataFrame()
    return pd.concat(daily_list, ignore_index=True)


def load_minute_panel_with_daily_fields(
    start_date: str,
    end_date: str,
    instruments: List[str],
    minute_fields: Optional[List[str]] = None,
    daily_fields: Optional[List[str]] = None,
    chunk_size: int = 100,
) -> pd.DataFrame:
    """
    加载分钟面板，并把日频因子库字段广播到分钟索引上。
    返回 DataFrame 列：date, time, instrument, [minute_fields], [daily_fields]
    """
    import dai

    if minute_fields is None:
        minute_fields = ["open", "high", "low", "close", "volume", "amount"]

    # 1. 读日频因子库
    daily_df = load_factor_library(start_date, end_date, instruments, daily_fields)

    # 2. 读分钟数据，并把列名加 m_ 前缀以匹配 RPN 词表
    all_dates = pd.date_range(start_date, end_date, freq="B").strftime("%Y-%m-%d").tolist()
    min_parts = []
    for date_str in all_dates:
        df_min = _query_minute_bars_day(date_str, instruments, minute_fields, chunk_size)
        if df_min.empty:
            continue
        df_min["datetime"] = pd.to_datetime(df_min["date"])
        df_min["date"] = df_min["datetime"].dt.strftime("%Y-%m-%d")
        df_min["time"] = df_min["datetime"].dt.hour * 100 + df_min["datetime"].dt.minute
        # 重命名为 m_*，与日频字段区分开
        rename_map = {c: f"m_{c}" for c in minute_fields}
        df_min = df_min.rename(columns=rename_map)
        min_parts.append(df_min[["date", "time", "instrument"] + list(rename_map.values())])

    if not min_parts:
        return pd.DataFrame()
    min_df = pd.concat(min_parts, ignore_index=True)

    # 3. 合并日频字段（左连接，按 date + instrument）
    if daily_fields is not None and daily_df is not None and not daily_df.empty:
        merged = min_df.merge(daily_df, on=["date", "instrument"], how="left")
    else:
        merged = min_df

    return merged


def merge_daily_and_hf(
    daily_df: pd.DataFrame,
    hf_df: pd.DataFrame,
    on: List[str] = None,
) -> pd.DataFrame:
    """合并日频因子库和日频高频特征。"""
    on = on or ["date", "instrument"]
    return daily_df.merge(hf_df, on=on, how="left")

# ================= Pipeline Helpers =================
"""
线下完整流程示例

1. 加载日频因子库 + 高频日频特征
2. 生成候选 RPN 表达式（随机 / 经验）
3. 本地评分、筛选、合成
4. 输出最终因子表达式 + 权重，供提交 notebook 使用
"""

import os
from pathlib import Path



VOCAB = Vocabulary()
EVALUATOR = RPNEvaluator(VOCAB)


def generate_random_candidates(vocab: Vocabulary, n: int = 200, max_len: int = 12, seed: int = 42) -> List[List[str]]:
    rng = np.random.default_rng(seed)
    candidates = []
    for _ in range(n):
        expr = vocab.sample_expression(rng, max_len=max_len)
        if expr not in candidates:
            candidates.append(expr)
    return candidates


MIN_KEEP_ICIR = 0.05  # 全样本 |ICIR| 低于该阈值的候选直接丢弃（不再回传大因子序列，防止父进程内存堆积），可按需调整
MAX_KEEP_FACTORS = 50  # 评估后最多保留的因子数（按 |ICIR| 取前 N 个），控制 factors_df 合并时的内存峰值
# 初筛淘汰统计的全局暂存：平台日志查看器会丢弃 print 行，诊断信息改由
# run_all_in_one 步骤 9 落盘为 eval_fail_stats.json，从工作目录取回。
EVAL_FAIL_STATS: Dict[str, int] = {}
EVAL_FAIL_MSGS: Dict[str, str] = {}


def _collect_unary_field_pairs(candidates: List[List[str]], vocab) -> Dict[Tuple[str, str], int]:
    """统计候选表达式中 "字段 token 紧跟一元算子" 的出现次数。

    这类子表达式只取决于 (字段, 算子)，与所在表达式无关，可全局只算一次后查表复用，
    避免几百条候选重复执行相同的 rolling/groupby 一元运算。
    """
    counts: Dict[Tuple[str, str], int] = {}
    for expr in candidates:
        for a, b in zip(expr, expr[1:]):
            if vocab.is_field(a) and b in vocab.OPERATORS and vocab.arity(b) == 1:
                counts[(a, b)] = counts.get((a, b), 0) + 1
    return counts


def _is_daily_unary_op(op: str) -> bool:
    """日频字段上可能合法的一元算子：分钟算子作用于日频字段必然报错，预计算时直接排除。"""
    return not (op.startswith("Ms_") or op.startswith("Day_"))


def _precompute_one_ctx(ev: RPNEvaluator, key: Tuple[str, str], df: pd.DataFrame) -> Optional[np.ndarray]:
    """在已建好的上下文上预计算单个 (字段, 一元算子)，返回 float32 数组或 None。"""
    field, op = key
    try:
        s = ev._apply_unary(op, df[field])
        if isinstance(s, pd.Series) and not s.dropna().empty:
            return s.to_numpy(dtype=np.float32)
    except Exception:
        pass
    return None


def _precompute_one(key: Tuple[str, str], payload: Dict) -> Tuple[Tuple[str, str], Optional[np.ndarray]]:
    """loky worker 入口：重建上下文后预计算单个组合。返回的数组同样走 memmap 回传。"""
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    df, _, _ = _build_eval_ctx(payload)
    ev = RPNEvaluator(VOCAB)
    return key, _precompute_one_ctx(ev, key, df)


def _precompute_unary_field_ops(
    pairs: Dict[Tuple[str, str], int],
    payload: Dict,
    n_jobs: int = -1,
    min_count: int = 2,
    max_precompute: int = 120,
) -> Tuple[List[Tuple[str, str]], Optional[np.ndarray]]:
    """对重复出现的 (字段, 一元算子) 组合各求值一次，返回 (keys, values)。

    values 为 float32 的 (n_rows, n_keys) 数组，列与 keys 对齐。
    只预计算出现 >= min_count 次、且对日频字段合法的组合（分钟算子必然报错，
    提前排除避免白算），并按出现次数限制最多 max_precompute 个。
    并行 worker 复用 payload 的 memmap 数组，传输开销可忽略。
    """
    keys = [k for k, c in pairs.items()
            if c >= min_count and k[0] in payload["columns"] and _is_daily_unary_op(k[1])]
    keys.sort(key=lambda k: pairs[k], reverse=True)
    keys = keys[:max_precompute]
    if not keys:
        return [], None

    if Parallel is not None and n_jobs != 1 and len(keys) > 4:
        results = Parallel(n_jobs=n_jobs, backend="loky", verbose=5, batch_size=4)(
            delayed(_precompute_one)(key, payload) for key in tqdm(keys, desc="预计算字段一元算子")
        )
    else:
        # 串行回退：上下文只建一次
        df_ctx, _, _ = _build_eval_ctx(payload)
        ev = RPNEvaluator(VOCAB)
        results = [(key, _precompute_one_ctx(ev, key, df_ctx))
                   for key in tqdm(keys, desc="预计算字段一元算子")]

    valid_keys = [k for k, arr in results if arr is not None]
    cols = [arr for _, arr in results if arr is not None]
    if not valid_keys:
        return [], None
    total_hits = sum(pairs[k] for k in valid_keys)
    print(f"预计算 {len(valid_keys)} 个 (字段, 算子) 组合，候选中共引用 {total_hits} 次")
    return valid_keys, np.stack(cols, axis=1)


def _build_eval_payload(
    df: pd.DataFrame,
    returns: pd.Series,
    pre_keys: List[Tuple[str, str]],
    pre_values: Optional[np.ndarray],
) -> Dict:
    """把评估所需数据打包成纯 numpy 载荷。

    loky 只对独立的大 numpy 数组做 memmap 零拷贝；带 MultiIndex 的 DataFrame
    pickle 后是普通字节流，每个 batch 都要完整序列化一次。拆成 values + 索引
    codes 之后，worker 直接从共享内存映射读取，传输开销接近零。
    """
    idx = df.index
    date_codes, date_uniques = pd.factorize(idx.get_level_values("date"))
    inst_codes, inst_uniques = pd.factorize(idx.get_level_values("instrument"))
    return {
        "values": df.to_numpy(dtype=np.float32),
        "columns": list(df.columns),
        "date_codes": date_codes.astype(np.int32),
        "inst_codes": inst_codes.astype(np.int32),
        "date_uniques": np.asarray(date_uniques),
        "inst_uniques": np.asarray(inst_uniques),
        "returns": returns.to_numpy(dtype=np.float32),
        "pre_keys": pre_keys,
        "pre_values": pre_values,
    }


def _build_eval_ctx(payload: Dict) -> Tuple[pd.DataFrame, pd.Series, Dict[Tuple[str, str], pd.Series]]:
    """从 numpy 载荷重建评估上下文（每个任务组只调用一次）。"""
    index = pd.MultiIndex.from_arrays(
        [
            payload["date_uniques"][payload["date_codes"]],
            payload["inst_uniques"][payload["inst_codes"]],
        ],
        names=["date", "instrument"],
    )
    df = pd.DataFrame(payload["values"], index=index, columns=payload["columns"])
    returns = pd.Series(payload["returns"], index=index)
    precomputed: Dict[Tuple[str, str], pd.Series] = {}
    if payload.get("pre_values") is not None:
        for j, key in enumerate(payload["pre_keys"]):
            precomputed[key] = pd.Series(payload["pre_values"][:, j], index=index)
    return df, returns, precomputed


def _eval_single_ctx(
    ev: RPNEvaluator,
    expr: List[str],
    df: pd.DataFrame,
    returns: pd.Series,
    precomputed: Dict[Tuple[str, str], pd.Series],
) -> Optional[Dict]:
    """单个候选表达式评估（在已建好的上下文上运行）。

    只回传指标（KB 级），不回传 145 万行的因子 Series（~11MB/条）。
    否则几百个通过初筛的候选在父进程汇总时会产生数 GB 内存峰值，内核直接被 OOM 杀掉。
    """
    name = "_".join(expr[1:-1])[:50]
    try:
        factor = ev.evaluate(expr, df, precomputed=precomputed)
        if factor is None or factor.dropna().empty:
            return {"_fail": "empty_factor"}
        metrics = compute_ic_metrics(compute_ic(factor, returns))
        if pd.isna(metrics["icir"]) or abs(metrics["icir"]) < MIN_KEEP_ICIR:
            return {"_fail": f"icir_below_{MIN_KEEP_ICIR}"}
        return {
            "name": name,
            "expr": expr,
            "ic_mean": metrics["ic_mean"],
            "icir": metrics["icir"],
            "ic_ratio": metrics["ic_ratio"],
        }
    except Exception as e:
        # 初筛静默死亡的统计标记：worker 进程写不了父进程计数器，改为回传失败原因，
        # 由父进程汇总打印；_fail_msg 每类只留第一条，载荷仍为 KB 级。
        return {"_fail": f"exception:{type(e).__name__}", "_fail_msg": str(e)[:200]}


def _eval_chunk(exprs: List[List[str]], payload: Dict) -> List[Optional[Dict]]:
    """一个任务评估一组候选：每组只重建一次上下文，摊薄 DataFrame 重建开销。"""
    warnings.filterwarnings("ignore", category=RuntimeWarning)  # loky worker 进程内也需屏蔽
    df, returns, precomputed = _build_eval_ctx(payload)
    ev = RPNEvaluator(VOCAB)
    return [_eval_single_ctx(ev, expr, df, returns, precomputed) for expr in exprs]


def _eval_single(expr: List[str], df: pd.DataFrame, returns: pd.Series) -> Optional[Dict]:
    """单个候选表达式评估（兼容旧接口）。"""
    warnings.filterwarnings("ignore", category=RuntimeWarning)  # loky worker 进程内也需屏蔽
    ev = RPNEvaluator(VOCAB)
    return _eval_single_ctx(ev, expr, df, returns, {})


def evaluate_candidates(
    candidates: List[List[str]],
    df: pd.DataFrame,
    returns: pd.Series,
    n_jobs: int = -1,
    eval_chunk_size: int = 8,
) -> Tuple[pd.DataFrame, Dict[str, List[str]]]:
    """评估候选表达式，返回 factors DataFrame 与 name->expr 映射。支持 joblib 并行。

    性能要点：
    1. 预计算重复出现的 (字段, 一元算子) 组合，表达式求值时查表复用；
    2. 并行载荷为纯 numpy 数组（loky memmap 零拷贝），不再逐 batch 序列化大 DataFrame；
    3. 手动按 eval_chunk_size 分块，每个任务重建一次上下文后评估整组。

    注意：并行载荷为 float32（筛选指标可能有末位精度差异）；Top 因子序列仍在
    父进程用原始 float64 数据重算，最终输出精度不受影响。
    """
    df = _to_series_index(df)
    returns = returns.reindex(df.index)

    # 1. 先打包 numpy 载荷（预计算 worker 复用同一份 memmap 数据）
    pairs = _collect_unary_field_pairs(candidates, VOCAB)
    payload = _build_eval_payload(df, returns, [], None)

    # 2. 并行预计算重复的 (字段, 一元算子) 组合，挂回载荷
    pre_keys, pre_values = _precompute_unary_field_ops(pairs, payload, n_jobs=n_jobs)
    payload["pre_keys"] = pre_keys
    payload["pre_values"] = pre_values
    del pre_values

    chunks = [
        candidates[i: i + eval_chunk_size]
        for i in range(0, len(candidates), eval_chunk_size)
    ]

    if Parallel is not None and n_jobs != 1 and len(chunks) > 2:
        # batch_size=1：任务已手动分块；payload 中的大数组由 loky memmap 共享
        chunked_results = Parallel(n_jobs=n_jobs, backend="loky", verbose=10, batch_size=1)(
            delayed(_eval_chunk)(chunk, payload) for chunk in tqdm(chunks, desc="评估候选因子")
        )
        results = [r for chunk_res in chunked_results for r in chunk_res]
        del chunked_results
    else:
        # 回退到串行（无 joblib 或显式 n_jobs=1）
        df_ctx, ret_ctx, pre_ctx = _build_eval_ctx(payload)
        ev = RPNEvaluator(VOCAB)
        results = [
            _eval_single_ctx(ev, expr, df_ctx, ret_ctx, pre_ctx)
            for expr in tqdm(candidates, desc="评估候选因子")
        ]
    del payload

    import gc

    print("并行评估完成，正在汇总结果...")
    scored = []  # (|icir|, name, expr) —— 注意：此处已无因子序列，内存开销可忽略
    fail_counter: Dict[str, int] = {}
    fail_msgs: Dict[str, str] = {}
    for r in results:
        if r is None:
            fail_counter["unknown"] = fail_counter.get("unknown", 0) + 1
            continue
        if "_fail" in r:
            reason = r["_fail"]
            fail_counter[reason] = fail_counter.get(reason, 0) + 1
            if "_fail_msg" in r and reason not in fail_msgs:
                fail_msgs[reason] = r["_fail_msg"]
            continue
        icir_abs = 0.0 if pd.isna(r["icir"]) else abs(r["icir"])
        scored.append((icir_abs, r["name"], r["expr"]))
    del results  # 尽早释放引用
    gc.collect()

    print(f"有效表达式: {len(scored)} / {len(candidates)}")
    if fail_counter:
        print(f"候选淘汰原因分布: {dict(sorted(fail_counter.items(), key=lambda kv: -kv[1]))}")
        for reason, msg in fail_msgs.items():
            print(f"  [{reason}] 示例: {msg}")
    EVAL_FAIL_STATS.clear()
    EVAL_FAIL_STATS.update(fail_counter)
    EVAL_FAIL_MSGS.clear()
    EVAL_FAIL_MSGS.update(fail_msgs)
    if not scored:
        return pd.DataFrame(), {}

    # 按 |ICIR| 降序截断，控制 factors_df 合并时的内存峰值
    scored.sort(key=lambda t: t[0], reverse=True)
    if len(scored) > MAX_KEEP_FACTORS:
        print(f"按 |ICIR| 截断保留前 {MAX_KEEP_FACTORS} 个（共 {len(scored)} 个有效）")
        scored = scored[:MAX_KEEP_FACTORS]
        gc.collect()

    # 父进程串行重算 Top 因子的序列（求值是确定性的，与 worker 中结果逐值一致；
    # 只重算 MAX_KEEP_FACTORS 个，耗时约几十秒，换取汇总阶段内存从 GB 级降到 MB 级）
    valid_factors = {}
    name_to_expr = {}
    ev = RPNEvaluator(VOCAB)
    for _, name, expr in tqdm(scored, desc="重算Top因子序列"):
        try:
            factor = ev.evaluate(expr, df)
            if factor is None or factor.dropna().empty:
                continue
            valid_factors[name] = factor
            name_to_expr[name] = expr
        except Exception:
            continue

    factors_df = pd.DataFrame(valid_factors)
    factors_df.index.names = ["date", "instrument"]
    del valid_factors, scored
    gc.collect()
    return factors_df, name_to_expr




def diagnose_long_short_sharpe(
    factor: pd.Series,
    returns: pd.Series,
    n_quantiles: int = 5,
    label: str = "因子",
) -> float:
    """计算并打印多空组合夏普的详细诊断信息，帮助排查 nan。"""
    df = pd.concat([factor.rename("factor"), returns.rename("return")], axis=1)

    def _daily_pnl(x):
        x = x.replace([np.inf, -np.inf], np.nan).dropna()  # 防御 inf（因子或收益异常值）
        if len(x) < n_quantiles * 2:
            return np.nan
        try:
            x["q"] = pd.qcut(x["factor"], n_quantiles, labels=False, duplicates="drop")
        except Exception:
            return np.nan
        long_ret = x[x["q"] == n_quantiles - 1]["return"].mean()
        short_ret = x[x["q"] == 0]["return"].mean()
        return long_ret - short_ret

    pnl = df.groupby(level="date", group_keys=False).apply(_daily_pnl)
    pnl_clean = pnl.dropna()

    # print(f"\n[{label}] 多空夏普诊断")
    # print(f"  总交易日: {len(pnl)}")
    # print(f"  有效多空日收益天数: {len(pnl_clean)}")
    # print(f"  覆盖率: {len(pnl_clean) / len(pnl) * 100:.1f}%" if len(pnl) else "  覆盖率: 0.0%")
    # print(f"  日均多空收益: {pnl_clean.mean():.6f}")
    # print(f"  日收益标准差: {pnl_clean.std():.6f}")

    if len(pnl_clean) < 5:
        print("  警告：有效日收益少于 5 天，无法计算夏普")
        return np.nan
    if pnl_clean.std() == 0:
        print("  警告：日收益标准差为 0，无法计算夏普")
        return np.nan

    sharpe = pnl_clean.mean() / pnl_clean.std() * np.sqrt(252)
    # print(f"  多空夏普 (年化): {sharpe:.4f}")
    return sharpe


# ================= 冻结配置（提交模式开关） =================
# 非 None 时 run_all_in_one 跳过候选生成/评估/筛选/权重拟合，直接用固化表达式+权重合成。
# 训练/筛选对分钟表历史覆盖敏感（自检浅表 vs 全量表会产出不同池子和权重），会导致平台
# check_lookahead 两次运行在 cutoff 前的因子值不一致、被误判为未来函数；冻结后 main()
# 是纯因果计算，任意两次运行输出逐位一致。想重新训练时把 FROZEN_CONFIG_NAME 设为 None 即可。
# 默认快照：2026-07-27 全量表主运行 5 因子池（权重由该次诊断 ICIR 归一化重建，万分位精度），
# 验证集 IC=0.0322, ICIR=0.5443, 多空夏普=+1.0229, stress_icir=0.5155（已提交公榜，得分 0.45）。
# 备选快照见 data/submit_config_archive_20260727_*.json（6/8 因子版夏普为负，勿用作合成）。
FROZEN_CONFIG = {
    "expressions": [
        ["BEG", "vwap_dev", "ABS", "ABS", "Ts_Max_5", "LOG", "debt_to_asset_lf", "sma_20", "5", "SUB", "If_Else", "SEP"],
        ["BEG", "afternoon_return", "SEP"],
        ["BEG", "volatility_5", "3", "MUL", "Ts_Mean_20", "beta_000300SH_22", "sma_20", "SUB", "ADD", "-3", "MIN", "SEP"],
        ["BEG", "roa_avg_ttm", "SEP"],
        ["BEG", "sma_20", "gross_profit_rate_ttm", "SUB", "SEP"],
    ],
    "selected": [
        "vwap_dev_ABS_ABS_Ts_Max_5_LOG_debt_to_asset_lf_sma",
        "afternoon_return",
        "volatility_5_3_MUL_Ts_Mean_20_beta_000300SH_22_sma",
        "roa_avg_ttm",
        "sma_20_gross_profit_rate_ttm_SUB",
    ],
    "weights": {
        "values": {
            "vwap_dev_ABS_ABS_Ts_Max_5_LOG_debt_to_asset_lf_sma": 0.28362913725758926,
            "afternoon_return": 0.23936133762762968,
            "volatility_5_3_MUL_Ts_Mean_20_beta_000300SH_22_sma": 0.13204961282806826,
            "roa_avg_ttm": 0.1830329610087028,
            "sma_20_gross_profit_rate_ttm_SUB": 0.16192695127801002,
        },
        "signs": {
            "vwap_dev_ABS_ABS_Ts_Max_5_LOG_debt_to_asset_lf_sma": -1,
            "afternoon_return": 1,
            "volatility_5_3_MUL_Ts_Mean_20_beta_000300SH_22_sma": 1,
            "roa_avg_ttm": -1,
            "sma_20_gross_profit_rate_ttm_SUB": -1,
        },
    },
}


# ================= 多提交矩阵（2026-07-28） =================
# 平台评分：Score_i = 0.3*A_i + 0.7*B_i，团队分 = max(所有提交)，提交无上限。
# A 项 = IC/ICIR/多空夏普/stress 在全体提交上的百分位均值；B 项 = 全局 Elastic Net
# （全体团队因子一起回归，60 日窗 20 日步长）中本因子权重稳定性 mean|w|/(std|w|+eps)
# 的百分位，权重恒 0（被 L1 压掉）则 B=0。含义：独特（与全球提交池低相关）且跨期
# 稳定的单因子 B 项最占便宜；合成因子 A 项高但与各家单因子共线、B 项吃亏。
# 故除合成保底外，为每个有训练期证据的因子准备单因子配置（权重恒 1.0，
# 符号 = 训练期 IC 符号，数据来自 2026-07-27 各次运行的 IC 粗筛诊断）。
# 用法：把 FROZEN_CONFIG_NAME 换成目标键再上传运行，即为一个新提交。
FROZEN_CONFIGS = {
    # ---- 合成保底（公榜 0.45）----
    "composite_5factor": FROZEN_CONFIG,
    # ---- 冻结 5 因子的单因子版 ----
    # 训练期 IC=-0.0206, ICIR=-0.4139（全场最强单因子）
    "vwap_dev_combo": {
        "expressions": [["BEG", "vwap_dev", "ABS", "ABS", "Ts_Max_5", "LOG", "debt_to_asset_lf", "sma_20", "5", "SUB", "If_Else", "SEP"]],
        "selected": ["vwap_dev_ABS_ABS_Ts_Max_5_LOG_debt_to_asset_lf_sma"],
        "weights": {"values": {"vwap_dev_ABS_ABS_Ts_Max_5_LOG_debt_to_asset_lf_sma": 1.0},
                    "signs": {"vwap_dev_ABS_ABS_Ts_Max_5_LOG_debt_to_asset_lf_sma": -1}},
    },
    # 训练期 IC=0.0144, ICIR=0.3493（HF 因子）。验证期 IC 衰减 -60%（0.0058），公榜 0.3407——衰减样本
    "afternoon_return": {
        "expressions": [["BEG", "afternoon_return", "SEP"]],
        "selected": ["afternoon_return"],
        "weights": {"values": {"afternoon_return": 1.0}, "signs": {"afternoon_return": 1}},
    },
    # 训练期 IC=0.0080, ICIR=0.1927
    "volatility_combo": {
        "expressions": [["BEG", "volatility_5", "3", "MUL", "Ts_Mean_20", "beta_000300SH_22", "sma_20", "SUB", "ADD", "-3", "MIN", "SEP"]],
        "selected": ["volatility_5_3_MUL_Ts_Mean_20_beta_000300SH_22_sma"],
        "weights": {"values": {"volatility_5_3_MUL_Ts_Mean_20_beta_000300SH_22_sma": 1.0},
                    "signs": {"volatility_5_3_MUL_Ts_Mean_20_beta_000300SH_22_sma": 1}},
    },
    # 训练期 IC=-0.0115, ICIR=-0.2671
    "roa_avg_ttm": {
        "expressions": [["BEG", "roa_avg_ttm", "SEP"]],
        "selected": ["roa_avg_ttm"],
        "weights": {"values": {"roa_avg_ttm": 1.0}, "signs": {"roa_avg_ttm": -1}},
    },
    # 训练期 IC=-0.0084, ICIR=-0.2363
    "sma_20_gross_sub": {
        "expressions": [["BEG", "sma_20", "gross_profit_rate_ttm", "SUB", "SEP"]],
        "selected": ["sma_20_gross_profit_rate_ttm_SUB"],
        "weights": {"values": {"sma_20_gross_profit_rate_ttm_SUB": 1.0},
                    "signs": {"sma_20_gross_profit_rate_ttm_SUB": -1}},
    },
    # ---- 手工/随机简单因子（2026-07-27 IC 粗筛诊断）----
    # 训练期 IC=0.0077, ICIR=0.1668（HF）
    "intraday_return_Cs_Zscore": {
        "expressions": [["BEG", "intraday_return", "Cs_Zscore", "SEP"]],
        "selected": ["intraday_return_Cs_Zscore"],
        "weights": {"values": {"intraday_return_Cs_Zscore": 1.0}, "signs": {"intraday_return_Cs_Zscore": 1}},
    },
    # 训练期 IC=0.0059, ICIR=0.1541（HF 资金流）
    "netflow_Cs_Zscore": {
        "expressions": [["BEG", "netflow_amount_rate_main", "Cs_Zscore", "SEP"]],
        "selected": ["netflow_amount_rate_main_Cs_Zscore"],
        "weights": {"values": {"netflow_amount_rate_main_Cs_Zscore": 1.0},
                    "signs": {"netflow_amount_rate_main_Cs_Zscore": 1}},
    },
    # 训练期 IC=-0.0076, ICIR=-0.1541
    "realized_vol_Cs_Rank_NEG": {
        "expressions": [["BEG", "realized_vol", "Cs_Rank", "NEG", "SEP"]],
        "selected": ["realized_vol_Cs_Rank_NEG"],
        "weights": {"values": {"realized_vol_Cs_Rank_NEG": 1.0}, "signs": {"realized_vol_Cs_Rank_NEG": -1}},
    },
    # 训练期 IC=0.0054, ICIR=0.1288（HF 盘口）
    "ob_imb_morning_MUL_Cs_Rank": {
        "expressions": [["BEG", "ob_imbalance_l5", "morning_return", "MUL", "Cs_Rank", "SEP"]],
        "selected": ["ob_imbalance_l5_morning_return_MUL_Cs_Rank"],
        "weights": {"values": {"ob_imbalance_l5_morning_return_MUL_Cs_Rank": 1.0},
                    "signs": {"ob_imbalance_l5_morning_return_MUL_Cs_Rank": 1}},
    },
    # 训练期 IC=0.0075, ICIR=0.1530
    "rsi_12": {
        "expressions": [["BEG", "rsi_12", "SEP"]],
        "selected": ["rsi_12"],
        "weights": {"values": {"rsi_12": 1.0}, "signs": {"rsi_12": 1}},
    },
    # 训练期 IC=-0.0071, ICIR=-0.1448
    "reversal_5": {
        "expressions": [["BEG", "reversal_5", "SEP"]],
        "selected": ["reversal_5"],
        "weights": {"values": {"reversal_5": 1.0}, "signs": {"reversal_5": -1}},
    },
    # 训练期 IC=-0.0061, ICIR=-0.1501
    "pb": {
        "expressions": [["BEG", "pb", "SEP"]],
        "selected": ["pb"],
        "weights": {"values": {"pb": 1.0}, "signs": {"pb": -1}},
    },
    # 训练期 IC=-0.0144, ICIR=-0.2769
    "total_market_cap": {
        "expressions": [["BEG", "total_market_cap", "SEP"]],
        "selected": ["total_market_cap"],
        "weights": {"values": {"total_market_cap": 1.0}, "signs": {"total_market_cap": -1}},
    },
    # ---- 栈深修复后 8 因子池的单因子版（2026-07-27 运行产物，存档见 data/ 8factors；
    #      符号 = 该次入池符号。微观结构含量高的优先提交）----
    # 训练期 IC=0.0402, ICIR=0.5517（全场 ICIR 最高单因子；验证 IC 0.0386 零衰减，夏普 -1.02 无碍）
    # 公榜已提交：0.8547（B 项近满分；当前团队分纪录）
    "qia_rsi_div_8f": {
        "expressions": [["BEG", "-10", "rsi_12", "Ts_Delay_1", "sma_20", "1", "quote_intensity_ask", "DIV", "DIV", "ADD", "DIV", "SEP"]],
        "selected": ["-10_rsi_12_Ts_Delay_1_sma_20_1_quote_intensity_ask"],
        "weights": {"values": {"-10_rsi_12_Ts_Delay_1_sma_20_1_quote_intensity_ask": 1.0},
                    "signs": {"-10_rsi_12_Ts_Delay_1_sma_20_1_quote_intensity_ask": 1}},
    },
    "rvol_morning_vwap_8f": {
        "expressions": [["BEG", "realized_vol", "rsi_12", "60", "morning_return", "vwap_dev", "Ts_Sum_5", "MIN", "MUL", "If_Else", "Ts_Std_5", "SEP"]],
        "selected": ["realized_vol_rsi_12_60_morning_return_vwap_dev_Ts_"],
        "weights": {"values": {"realized_vol_rsi_12_60_morning_return_vwap_dev_Ts_": 1.0},
                    "signs": {"realized_vol_rsi_12_60_morning_return_vwap_dev_Ts_": 1}},
    },
    "roe_ob_imb_8f": {
        "expressions": [["BEG", "roe_avg_ttm", "Ts_Rank_5", "ob_imbalance_l5", "SIGN", "ob_imbalance_l5", "roa_avg_ttm", "ADD", "If_Else", "atr_14", "MAX", "SEP"]],
        "selected": ["roe_avg_ttm_Ts_Rank_5_ob_imbalance_l5_SIGN_ob_imba"],
        "weights": {"values": {"roe_avg_ttm_Ts_Rank_5_ob_imbalance_l5_SIGN_ob_imba": 1.0},
                    "signs": {"roe_avg_ttm_Ts_Rank_5_ob_imbalance_l5_SIGN_ob_imba": -1}},
    },
    "afternoon_pe_roe_8f": {
        "expressions": [["BEG", "afternoon_return", "list_days", "atr_14", "SUB", "pe_ttm", "roe_avg_ttm", "If_Else", "Cs_Rank", "NEG", "MAX", "SEP"]],
        "selected": ["afternoon_return_list_days_atr_14_SUB_pe_ttm_roe_a"],
        "weights": {"values": {"afternoon_return_list_days_atr_14_SUB_pe_ttm_roe_a": 1.0},
                    "signs": {"afternoon_return_list_days_atr_14_SUB_pe_ttm_roe_a": 1}},
    },
    "roe_mcap_netflow_8f": {
        "expressions": [["BEG", "60", "roe_avg_ttm", "total_market_cap", "netflow_amount_rate_main", "Ts_Mean_5", "sma_20", "ADD", "MIN", "MAX", "MUL", "SEP"]],
        "selected": ["60_roe_avg_ttm_total_market_cap_netflow_amount_rat"],
        "weights": {"values": {"60_roe_avg_ttm_total_market_cap_netflow_amount_rat": 1.0},
                    "signs": {"60_roe_avg_ttm_total_market_cap_netflow_amount_rat": -1}},
    },
    "floatcap_cci_8f": {
        "expressions": [["BEG", "20", "float_market_cap", "Cs_Rank", "1", "atr_14", "cci_14", "MIN", "MAX", "MAX", "ADD", "SEP"]],
        "selected": ["20_float_market_cap_Cs_Rank_1_atr_14_cci_14_MIN_MA"],
        "weights": {"values": {"20_float_market_cap_Cs_Rank_1_atr_14_cci_14_MIN_MA": 1.0},
                    "signs": {"20_float_market_cap_Cs_Rank_1_atr_14_cci_14_MIN_MA": -1}},
    },
    "pb_spread_vol_8f": {
        "expressions": [["BEG", "pb", "sma_20", "spread_l1_mean", "1", "volatility_5", "Ts_Sum_5", "DIV", "ADD", "ADD", "MUL", "SEP"]],
        "selected": ["pb_sma_20_spread_l1_mean_1_volatility_5_Ts_Sum_5_D"],
        "weights": {"values": {"pb_sma_20_spread_l1_mean_1_volatility_5_Ts_Sum_5_D": 1.0},
                    "signs": {"pb_sma_20_spread_l1_mean_1_volatility_5_Ts_Sum_5_D": -1}},
    },
    "ema_netbuy_qia_8f": {
        "expressions": [["BEG", "ema_20", "sma_20", "net_active_buy_amount_main", "Ts_Sum_5", "SIGN", "quote_intensity_ask", "Ts_Mean_10", "ADD", "SUB", "ADD", "SEP"]],
        "selected": ["ema_20_sma_20_net_active_buy_amount_main_Ts_Sum_5_"],
        "weights": {"values": {"ema_20_sma_20_net_active_buy_amount_main_Ts_Sum_5_": 1.0},
                    "signs": {"ema_20_sma_20_net_active_buy_amount_main_Ts_Sum_5_": -1}},
    },
    # ---- 2026-07-28 第二批：候选扩容后 8 因子池 v2（验证集 IC=0.0483, ICIR=0.604,
    #      夏普=+0.26 转正, stress=0.463；配置来自 16:32 运行的 submit_config.json）
    #      公榜 0.23113：验证 IC 全场最高但 B 项≈0.07——成员因子（含 qia）已在池，
    #      合成无增量贡献被 L1 压死。实证结论：合成路线不值得提交，只交单因子 ----
    "composite_8factor_v2": {
        "expressions": [
            ["BEG", "-10", "rsi_12", "Ts_Delay_1", "sma_20", "1", "quote_intensity_ask", "DIV", "DIV", "ADD", "DIV", "SEP"],
            ["BEG", "afternoon_return", "list_days", "atr_14", "SUB", "pe_ttm", "roe_avg_ttm", "If_Else", "Cs_Rank", "NEG", "MAX", "SEP"],
            ["BEG", "sma_20", "sma_20", "atr_14", "Ts_Rank_20", "ob_imbalance_l5", "Ts_Max_5", "Ts_Min_5", "DIV", "SUB", "MUL", "SEP"],
            ["BEG", "realized_vol", "atr_14", "list_days", "-2", "kdj_k_9_3_3", "Ts_Rank_20", "ADD", "ADD", "SUB", "SUB", "SEP"],
            ["BEG", "float_market_cap", "netflow_amount_rate_main", "intraday_return", "MUL", "SUB", "-5", "MUL", "intraday_return", "Ts_Std_20", "MIN", "SEP"],
            ["BEG", "roe_avg_ttm", "Ts_Rank_5", "ob_imbalance_l5", "SIGN", "ob_imbalance_l5", "roa_avg_ttm", "ADD", "If_Else", "atr_14", "MAX", "SEP"],
            ["BEG", "ema_20", "vwap_dev", "netflow_amount_main", "total_market_cap", "pe_ttm", "DIV", "SIGN", "MIN", "If_Else", "Ts_Std_5", "SEP"],
            ["BEG", "quote_intensity_bid", "SIGN", "NEG", "current_ratio_lf", "realized_vol", "morning_return", "DIV", "MAX", "Ts_Std_20", "MIN", "SEP"],
        ],
        "selected": [
            "-10_rsi_12_Ts_Delay_1_sma_20_1_quote_intensity_ask",
            "afternoon_return_list_days_atr_14_SUB_pe_ttm_roe_a",
            "sma_20_sma_20_atr_14_Ts_Rank_20_ob_imbalance_l5_Ts",
            "realized_vol_atr_14_list_days_-2_kdj_k_9_3_3_Ts_Ra",
            "float_market_cap_netflow_amount_rate_main_intraday",
            "roe_avg_ttm_Ts_Rank_5_ob_imbalance_l5_SIGN_ob_imba",
            "ema_20_vwap_dev_netflow_amount_main_total_market_c",
            "quote_intensity_bid_SIGN_NEG_current_ratio_lf_real",
        ],
        "weights": {
            "values": {
                "-10_rsi_12_Ts_Delay_1_sma_20_1_quote_intensity_ask": 0.16122338358714308,
                "afternoon_return_list_days_atr_14_SUB_pe_ttm_roe_a": 0.1030095348870123,
                "sma_20_sma_20_atr_14_Ts_Rank_20_ob_imbalance_l5_Ts": 0.13124377515536279,
                "realized_vol_atr_14_list_days_-2_kdj_k_9_3_3_Ts_Ra": 0.11938465523970143,
                "float_market_cap_netflow_amount_rate_main_intraday": 0.11650158427625473,
                "roe_avg_ttm_Ts_Rank_5_ob_imbalance_l5_SIGN_ob_imba": 0.10236327563409953,
                "ema_20_vwap_dev_netflow_amount_main_total_market_c": 0.14606552265237904,
                "quote_intensity_bid_SIGN_NEG_current_ratio_lf_real": 0.12020826856804712,
            },
            "signs": {
                "-10_rsi_12_Ts_Delay_1_sma_20_1_quote_intensity_ask": 1,
                "afternoon_return_list_days_atr_14_SUB_pe_ttm_roe_a": 1,
                "sma_20_sma_20_atr_14_Ts_Rank_20_ob_imbalance_l5_Ts": 1,
                "realized_vol_atr_14_list_days_-2_kdj_k_9_3_3_Ts_Ra": 1,
                "float_market_cap_netflow_amount_rate_main_intraday": 1,
                "roe_avg_ttm_Ts_Rank_5_ob_imbalance_l5_SIGN_ob_imba": -1,
                "ema_20_vwap_dev_netflow_amount_main_total_market_c": 1,
                "quote_intensity_bid_SIGN_NEG_current_ratio_lf_real": 1,
            },
        },
    },
    # ---- v2 池的新单因子（另 3 个成员 qia / afternoon_pe_roe_8f / roe_ob_imb_8f 已在册）----
    # 训练期 IC=0.0285, ICIR=0.4491
    "sma_atr_obimb_v2": {
        "expressions": [["BEG", "sma_20", "sma_20", "atr_14", "Ts_Rank_20", "ob_imbalance_l5", "Ts_Max_5", "Ts_Min_5", "DIV", "SUB", "MUL", "SEP"]],
        "selected": ["sma_20_sma_20_atr_14_Ts_Rank_20_ob_imbalance_l5_Ts"],
        "weights": {"values": {"sma_20_sma_20_atr_14_Ts_Rank_20_ob_imbalance_l5_Ts": 1.0},
                    "signs": {"sma_20_sma_20_atr_14_Ts_Rank_20_ob_imbalance_l5_Ts": 1}},
    },
    # 训练期 IC=0.0173, ICIR=0.4085
    "rvol_kdj_v2": {
        "expressions": [["BEG", "realized_vol", "atr_14", "list_days", "-2", "kdj_k_9_3_3", "Ts_Rank_20", "ADD", "ADD", "SUB", "SUB", "SEP"]],
        "selected": ["realized_vol_atr_14_list_days_-2_kdj_k_9_3_3_Ts_Ra"],
        "weights": {"values": {"realized_vol_atr_14_list_days_-2_kdj_k_9_3_3_Ts_Ra": 1.0},
                    "signs": {"realized_vol_atr_14_list_days_-2_kdj_k_9_3_3_Ts_Ra": 1}},
    },
    # 训练期 IC=0.0181, ICIR=0.3987
    "floatcap_netflow_v2": {
        "expressions": [["BEG", "float_market_cap", "netflow_amount_rate_main", "intraday_return", "MUL", "SUB", "-5", "MUL", "intraday_return", "Ts_Std_20", "MIN", "SEP"]],
        "selected": ["float_market_cap_netflow_amount_rate_main_intraday"],
        "weights": {"values": {"float_market_cap_netflow_amount_rate_main_intraday": 1.0},
                    "signs": {"float_market_cap_netflow_amount_rate_main_intraday": 1}},
    },
    # 训练期 IC=0.0280, ICIR=0.4998（v2 池中最强单因子）
    "ema_vwap_netflow_v2": {
        "expressions": [["BEG", "ema_20", "vwap_dev", "netflow_amount_main", "total_market_cap", "pe_ttm", "DIV", "SIGN", "MIN", "If_Else", "Ts_Std_5", "SEP"]],
        "selected": ["ema_20_vwap_dev_netflow_amount_main_total_market_c"],
        "weights": {"values": {"ema_20_vwap_dev_netflow_amount_main_total_market_c": 1.0},
                    "signs": {"ema_20_vwap_dev_netflow_amount_main_total_market_c": 1}},
    },
    # 训练期 IC=0.0158, ICIR=0.4113
    "qib_sign_v2": {
        "expressions": [["BEG", "quote_intensity_bid", "SIGN", "NEG", "current_ratio_lf", "realized_vol", "morning_return", "DIV", "MAX", "Ts_Std_20", "MIN", "SEP"]],
        "selected": ["quote_intensity_bid_SIGN_NEG_current_ratio_lf_real"],
        "weights": {"values": {"quote_intensity_bid_SIGN_NEG_current_ratio_lf_real": 1.0},
                    "signs": {"quote_intensity_bid_SIGN_NEG_current_ratio_lf_real": 1}},
    },
    # ---- 2026-07-28 第三批：candidate_ic_table 中 |ICIR|>=0.39 的其余候选
    #      （完整 50 条存档 data/candidate_ic_table_20260728.json；已在册的 11 条不重复收）----
    # 训练期 IC=-0.0333, ICIR=-0.5447（全场 #2，盘口强度类）
    "qib_ema_macd_v2": {
        "expressions": [["BEG", "quote_intensity_bid", "ema_20", "MUL", "Ts_Sum_5", "macd_hist_12_26_9", "Ts_Std_5", "intraday_return", "Ts_Max_5", "DIV", "MAX", "SEP"]],
        "selected": ["quote_intensity_bid_ema_20_MUL_Ts_Sum_5_macd_hist_"],
        "weights": {"values": {"quote_intensity_bid_ema_20_MUL_Ts_Sum_5_macd_hist_": 1.0},
                    "signs": {"quote_intensity_bid_ema_20_MUL_Ts_Sum_5_macd_hist_": -1}},
    },
    # 训练期 IC=0.0335, ICIR=0.5036（全场 #3）
    "atr5_morning_macd_v2": {
        "expressions": [["BEG", "-5", "atr_14", "morning_return", "macd_dea_12_26_9", "Ts_Mean_5", "roe_avg_ttm", "MIN", "MIN", "MAX", "SUB", "SEP"]],
        "selected": ["-5_atr_14_morning_return_macd_dea_12_26_9_Ts_Mean_"],
        "weights": {"values": {"-5_atr_14_morning_return_macd_dea_12_26_9_Ts_Mean_": 1.0},
                    "signs": {"-5_atr_14_morning_return_macd_dea_12_26_9_Ts_Mean_": 1}},
    },
    # 训练期 IC=-0.0267, ICIR=-0.4743
    "atr_intraday_rsi_v2": {
        "expressions": [["BEG", "atr_14", "5", "intraday_return", "afternoon_return", "rsi_12", "Ts_Mean_5", "MIN", "MIN", "MIN", "SUB", "SEP"]],
        "selected": ["atr_14_5_intraday_return_afternoon_return_rsi_12_T"],
        "weights": {"values": {"atr_14_5_intraday_return_afternoon_return_rsi_12_T": 1.0},
                    "signs": {"atr_14_5_intraday_return_afternoon_return_rsi_12_T": -1}},
    },
    # 训练期 IC=-0.0199, ICIR=-0.4726
    "ema_debt_netflow_v2": {
        "expressions": [["BEG", "ema_20", "debt_to_asset_lf", "netflow_amount_rate_main", "beta_000300SH_22", "roe_avg_ttm", "SIGN", "DIV", "MUL", "MUL", "SUB", "SEP"]],
        "selected": ["ema_20_debt_to_asset_lf_netflow_amount_rate_main_b"],
        "weights": {"values": {"ema_20_debt_to_asset_lf_netflow_amount_rate_main_b": 1.0},
                    "signs": {"ema_20_debt_to_asset_lf_netflow_amount_rate_main_b": -1}},
    },
    # 训练期 IC=-0.0202, ICIR=-0.4708
    "gross_atr_obimb_v2": {
        "expressions": [["BEG", "gross_profit_rate_ttm", "atr_14", "ob_imbalance_l5", "Ts_Rank_20", "current_ratio_lf", "reversal_5", "If_Else", "Cs_Zscore", "MAX", "MAX", "SEP"]],
        "selected": ["gross_profit_rate_ttm_atr_14_ob_imbalance_l5_Ts_Ra"],
        "weights": {"values": {"gross_profit_rate_ttm_atr_14_ob_imbalance_l5_Ts_Ra": 1.0},
                    "signs": {"gross_profit_rate_ttm_atr_14_ob_imbalance_l5_Ts_Ra": -1}},
    },
    # 训练期 IC=0.0238, ICIR=0.4689
    "ema_neglog_mom_v2": {
        "expressions": [["BEG", "60", "ema_20", "NEG", "LOG", "3", "ADD", "momentum_5", "Ts_Mean_10", "SUB", "DIV", "SEP"]],
        "selected": ["60_ema_20_NEG_LOG_3_ADD_momentum_5_Ts_Mean_10_SUB_"],
        "weights": {"values": {"60_ema_20_NEG_LOG_3_ADD_momentum_5_Ts_Mean_10_SUB_": 1.0},
                    "signs": {"60_ema_20_NEG_LOG_3_ADD_momentum_5_Ts_Mean_10_SUB_": 1}},
    },
    # 训练期 IC=0.0410, ICIR=0.4513（全场 IC 最高）
    "ema_intraday_pe_v2": {
        "expressions": [["BEG", "ema_20", "intraday_return", "pe_ttm", "macd_dea_12_26_9", "10", "MIN", "Ts_Rank_20", "SUB", "If_Else", "Ts_Std_20", "SEP"]],
        "selected": ["ema_20_intraday_return_pe_ttm_macd_dea_12_26_9_10_"],
        "weights": {"values": {"ema_20_intraday_return_pe_ttm_macd_dea_12_26_9_10_": 1.0},
                    "signs": {"ema_20_intraday_return_pe_ttm_macd_dea_12_26_9_10_": 1}},
    },
    # 训练期 IC=-0.0316, ICIR=-0.4389
    "atr_rev_pe_roa_v2": {
        "expressions": [["BEG", "atr_14", "reversal_5", "pe_ttm", "SUB", "2", "roa_avg_ttm", "Ts_Mean_5", "ADD", "If_Else", "NEG", "SEP"]],
        "selected": ["atr_14_reversal_5_pe_ttm_SUB_2_roa_avg_ttm_Ts_Mean"],
        "weights": {"values": {"atr_14_reversal_5_pe_ttm_SUB_2_roa_avg_ttm_Ts_Mean": 1.0},
                    "signs": {"atr_14_reversal_5_pe_ttm_SUB_2_roa_avg_ttm_Ts_Mean": -1}},
    },
    # 训练期 IC=-0.0233, ICIR=-0.4355
    "sma_intraday_cci_v2": {
        "expressions": [["BEG", "sma_20", "intraday_return", "Ts_Std_20", "cci_14", "Ts_Mean_10", "Cs_Rank", "Ts_Zscore_20", "Cs_Rank", "MIN", "MIN", "SEP"]],
        "selected": ["sma_20_intraday_return_Ts_Std_20_cci_14_Ts_Mean_10"],
        "weights": {"values": {"sma_20_intraday_return_Ts_Std_20_cci_14_Ts_Mean_10": 1.0},
                    "signs": {"sma_20_intraday_return_Ts_Std_20_cci_14_Ts_Mean_10": -1}},
    },
    # 训练期 IC=-0.0195, ICIR=-0.4353
    "netbuy_std_mcap_v2": {
        "expressions": [["BEG", "net_active_buy_amount_main", "Ts_Std_20", "total_market_cap", "volatility_5", "ps_ttm", "-1", "MAX", "SUB", "SUB", "MIN", "SEP"]],
        "selected": ["net_active_buy_amount_main_Ts_Std_20_total_market_"],
        "weights": {"values": {"net_active_buy_amount_main_Ts_Std_20_total_market_": 1.0},
                    "signs": {"net_active_buy_amount_main_Ts_Std_20_total_market_": -1}},
    },
    # 训练期 IC=-0.0207, ICIR=-0.4157
    "sma_netprofit_ema_v2": {
        "expressions": [["BEG", "sma_20", "net_profit_rate_ttm", "ema_20", "intraday_return", "Ts_Mean_5", "roa_avg_ttm", "MIN", "MAX", "MAX", "MIN", "SEP"]],
        "selected": ["sma_20_net_profit_rate_ttm_ema_20_intraday_return_"],
        "weights": {"values": {"sma_20_net_profit_rate_ttm_ema_20_intraday_return_": 1.0},
                    "signs": {"sma_20_net_profit_rate_ttm_ema_20_intraday_return_": -1}},
    },
    # 训练期 IC=0.0206, ICIR=0.4143
    "qib60_ps_min_v2": {
        "expressions": [["BEG", "60", "quote_intensity_bid", "ADD", "sma_20", "ps_ttm", "Ts_Min_5", "ABS", "Cs_Rank", "If_Else", "NEG", "SEP"]],
        "selected": ["60_quote_intensity_bid_ADD_sma_20_ps_ttm_Ts_Min_5_"],
        "weights": {"values": {"60_quote_intensity_bid_ADD_sma_20_ps_ttm_Ts_Min_5_": 1.0},
                    "signs": {"60_quote_intensity_bid_ADD_sma_20_ps_ttm_Ts_Min_5_": 1}},
    },
    # 训练期 IC=-0.0206, ICIR=-0.4139
    "sma60_floatcap_v2": {
        "expressions": [["BEG", "sma_20", "sma_20", "60", "3", "float_market_cap", "Cs_Rank", "MAX", "SUB", "MAX", "MUL", "SEP"]],
        "selected": ["sma_20_sma_20_60_3_float_market_cap_Cs_Rank_MAX_SU"],
        "weights": {"values": {"sma_20_sma_20_60_3_float_market_cap_Cs_Rank_MAX_SU": 1.0},
                    "signs": {"sma_20_sma_20_60_3_float_market_cap_Cs_Rank_MAX_SU": -1}},
    },
    # 训练期 IC=-0.0245, ICIR=-0.4136
    "ema_kdj_netbuy_v2": {
        "expressions": [["BEG", "ema_20", "kdj_d_9_3_3", "Ts_Mean_5", "Ts_Rank_20", "net_active_buy_amount_main", "spread_l1_mean", "SUB", "Cs_Rank", "SUB", "MAX", "SEP"]],
        "selected": ["ema_20_kdj_d_9_3_3_Ts_Mean_5_Ts_Rank_20_net_active"],
        "weights": {"values": {"ema_20_kdj_d_9_3_3_Ts_Mean_5_Ts_Rank_20_net_active": 1.0},
                    "signs": {"ema_20_kdj_d_9_3_3_Ts_Mean_5_Ts_Rank_20_net_active": -1}},
    },
    # 训练期 IC=0.0205, ICIR=0.4127
    "ema_rvol_rank_v2": {
        "expressions": [["BEG", "ema_20", "-10", "-5", "realized_vol", "Ts_Mean_5", "ABS", "Ts_Rank_5", "MIN", "MIN", "DIV", "SEP"]],
        "selected": ["ema_20_-10_-5_realized_vol_Ts_Mean_5_ABS_Ts_Rank_5"],
        "weights": {"values": {"ema_20_-10_-5_realized_vol_Ts_Mean_5_ABS_Ts_Rank_5": 1.0},
                    "signs": {"ema_20_-10_-5_realized_vol_Ts_Mean_5_ABS_Ts_Rank_5": 1}},
    },
    # 训练期 IC=-0.0205, ICIR=-0.4122
    "morning_sign_ema_v2": {
        "expressions": [["BEG", "morning_return", "Cs_Rank", "SIGN", "ema_20", "Ts_Delay_1", "-3", "netflow_amount_main", "DIV", "If_Else", "Ts_Sum_5", "SEP"]],
        "selected": ["morning_return_Cs_Rank_SIGN_ema_20_Ts_Delay_1_-3_n"],
        "weights": {"values": {"morning_return_Cs_Rank_SIGN_ema_20_Ts_Delay_1_-3_n": 1.0},
                    "signs": {"morning_return_Cs_Rank_SIGN_ema_20_Ts_Delay_1_-3_n": -1}},
    },
    # 训练期 IC=-0.0187, ICIR=-0.4090
    "sma_kdj_float_v2": {
        "expressions": [["BEG", "sma_20", "Ts_Max_5", "Ts_Min_5", "kdj_d_9_3_3", "kdj_d_9_3_3", "float_market_cap", "MIN", "Ts_Zscore_20", "MIN", "SUB", "SEP"]],
        "selected": ["sma_20_Ts_Max_5_Ts_Min_5_kdj_d_9_3_3_kdj_d_9_3_3_f"],
        "weights": {"values": {"sma_20_Ts_Max_5_Ts_Min_5_kdj_d_9_3_3_kdj_d_9_3_3_f": 1.0},
                    "signs": {"sma_20_Ts_Max_5_Ts_Min_5_kdj_d_9_3_3_kdj_d_9_3_3_f": -1}},
    },
    # 训练期 IC=-0.0206, ICIR=-0.4056
    "sma_ema_gross_v2": {
        "expressions": [["BEG", "sma_20", "ema_20", "Ts_Max_5", "gross_profit_rate_ttm", "Ts_Mean_10", "macd_diff_12_26_9", "Ts_Mean_10", "SUB", "SUB", "MAX", "SEP"]],
        "selected": ["sma_20_ema_20_Ts_Max_5_gross_profit_rate_ttm_Ts_Me"],
        "weights": {"values": {"sma_20_ema_20_Ts_Max_5_gross_profit_rate_ttm_Ts_Me": 1.0},
                    "signs": {"sma_20_ema_20_Ts_Max_5_gross_profit_rate_ttm_Ts_Me": -1}},
    },
    # 训练期 IC=-0.0150, ICIR=-0.4043
    "obimb_curr_mcap_v2": {
        "expressions": [["BEG", "ob_imbalance_l5", "current_ratio_lf", "total_market_cap", "ema_20", "Ts_Max_5", "Cs_Zscore", "Ts_Std_5", "MAX", "ADD", "ADD", "SEP"]],
        "selected": ["ob_imbalance_l5_current_ratio_lf_total_market_cap_"],
        "weights": {"values": {"ob_imbalance_l5_current_ratio_lf_total_market_cap_": 1.0},
                    "signs": {"ob_imbalance_l5_current_ratio_lf_total_market_cap_": -1}},
    },
    # 训练期 IC=-0.0164, ICIR=-0.4024
    "mcap_cci_qia_v2": {
        "expressions": [["BEG", "total_market_cap", "SIGN", "cci_14", "quote_intensity_ask", "float_market_cap", "MAX", "-2", "If_Else", "Cs_Rank", "SUB", "SEP"]],
        "selected": ["total_market_cap_SIGN_cci_14_quote_intensity_ask_f"],
        "weights": {"values": {"total_market_cap_SIGN_cci_14_quote_intensity_ask_f": 1.0},
                    "signs": {"total_market_cap_SIGN_cci_14_quote_intensity_ask_f": -1}},
    },
    # 训练期 IC=-0.0199, ICIR=-0.3988
    "spread_macd60_v2": {
        "expressions": [["BEG", "sma_20", "0", "60", "spread_l1_mean", "macd_diff_12_26_9", "Ts_Std_5", "MIN", "SUB", "SUB", "ADD", "SEP"]],
        "selected": ["sma_20_0_60_spread_l1_mean_macd_diff_12_26_9_Ts_St"],
        "weights": {"values": {"sma_20_0_60_spread_l1_mean_macd_diff_12_26_9_Ts_St": 1.0},
                    "signs": {"sma_20_0_60_spread_l1_mean_macd_diff_12_26_9_Ts_St": -1}},
    },
    # 训练期 IC=-0.0192, ICIR=-0.3984
    "gross_std_roe_v2": {
        "expressions": [["BEG", "gross_profit_rate_ttm", "Ts_Std_20", "sma_20", "roe_avg_ttm", "ps_ttm", "NEG", "Ts_Mean_5", "MUL", "MIN", "MUL", "SEP"]],
        "selected": ["gross_profit_rate_ttm_Ts_Std_20_sma_20_roe_avg_ttm"],
        "weights": {"values": {"gross_profit_rate_ttm_Ts_Std_20_sma_20_roe_avg_ttm": 1.0},
                    "signs": {"gross_profit_rate_ttm_Ts_Std_20_sma_20_roe_avg_ttm": -1}},
    },
    # 训练期 IC=0.0157, ICIR=0.3960
    "vwap_vol_gross_v2": {
        "expressions": [["BEG", "vwap_dev", "vwap_dev", "volatility_5", "volatility_5", "gross_profit_rate_ttm", "NEG", "ADD", "SUB", "SUB", "MIN", "SEP"]],
        "selected": ["vwap_dev_vwap_dev_volatility_5_volatility_5_gross_"],
        "weights": {"values": {"vwap_dev_vwap_dev_volatility_5_volatility_5_gross_": 1.0},
                    "signs": {"vwap_dev_vwap_dev_volatility_5_volatility_5_gross_": 1}},
    },
    # 训练期 IC=-0.0199, ICIR=-0.3942
    "netprofit_ema_macd_v2": {
        "expressions": [["BEG", "-2", "net_profit_rate_ttm", "ema_20", "Ts_Sum_5", "macd_dea_12_26_9", "2", "SUB", "ADD", "MIN", "ADD", "SEP"]],
        "selected": ["-2_net_profit_rate_ttm_ema_20_Ts_Sum_5_macd_dea_12"],
        "weights": {"values": {"-2_net_profit_rate_ttm_ema_20_Ts_Sum_5_macd_dea_12": 1.0},
                    "signs": {"-2_net_profit_rate_ttm_ema_20_Ts_Sum_5_macd_dea_12": -1}},
    },
    # ---- 2026-07-28 第四批：n_random=1500 新候选（存档 data/candidate_ic_table_20260728_n1500.json；
    #      与第二/三批重合的不重复收。前 6 个 |ICIR|>0.55 为新头部，插队优先提交）----
    # 训练期 IC=0.0561, ICIR=0.6837（全场 IC/ICIR 双料第一）
    "atr_atr_netflow_v3": {
        "expressions": [["BEG", "atr_14", "atr_14", "20", "60", "netflow_amount_main", "DIV", "Ts_Max_5", "SUB", "MUL", "MIN", "SEP"]],
        "selected": ["atr_14_atr_14_20_60_netflow_amount_main_DIV_Ts_Max"],
        "weights": {"values": {"atr_14_atr_14_20_60_netflow_amount_main_DIV_Ts_Max": 1.0},
                    "signs": {"atr_14_atr_14_20_60_netflow_amount_main_DIV_Ts_Max": 1}},
    },
    # 训练期 IC=-0.0405, ICIR=-0.6814（#2）
    "netprofit_ema_rvol_v3": {
        "expressions": [["BEG", "net_profit_rate_ttm", "3", "SUB", "ema_20", "MAX", "realized_vol", "reversal_5", "Ts_Mean_20", "DIV", "MAX", "SEP"]],
        "selected": ["net_profit_rate_ttm_3_SUB_ema_20_MAX_realized_vol_"],
        "weights": {"values": {"net_profit_rate_ttm_3_SUB_ema_20_MAX_realized_vol_": 1.0},
                    "signs": {"net_profit_rate_ttm_3_SUB_ema_20_MAX_realized_vol_": -1}},
    },
    # 训练期 IC=-0.0559, ICIR=-0.6241（#3，IC 绝对值第二）
    "atr60_netflow_debt_v3": {
        "expressions": [["BEG", "60", "atr_14", "10", "netflow_amount_main", "debt_to_asset_lf", "Cs_Rank", "DIV", "MAX", "MUL", "DIV", "SEP"]],
        "selected": ["60_atr_14_10_netflow_amount_main_debt_to_asset_lf_"],
        "weights": {"values": {"60_atr_14_10_netflow_amount_main_debt_to_asset_lf_": 1.0},
                    "signs": {"60_atr_14_10_netflow_amount_main_debt_to_asset_lf_": -1}},
    },
    # 训练期 IC=0.0416, ICIR=0.5956（#4）
    "mom_morning_delay_v3": {
        "expressions": [["BEG", "momentum_5", "Ts_Delay_5", "SIGN", "morning_return", "Ts_Delay_5", "Ts_Std_20", "ema_20", "SIGN", "SUB", "ADD", "SEP"]],
        "selected": ["momentum_5_Ts_Delay_5_SIGN_morning_return_Ts_Delay"],
        "weights": {"values": {"momentum_5_Ts_Delay_5_SIGN_morning_return_Ts_Delay": 1.0},
                    "signs": {"momentum_5_Ts_Delay_5_SIGN_morning_return_Ts_Delay": 1}},
    },
    # 训练期 IC=-0.0512, ICIR=-0.5875（#5，盘口类）
    "obimb_sma_vwap_v3": {
        "expressions": [["BEG", "ob_imbalance_l5", "sma_20", "Ts_Max_5", "sma_20", "MAX", "60", "vwap_dev", "SUB", "MUL", "MAX", "SEP"]],
        "selected": ["ob_imbalance_l5_sma_20_Ts_Max_5_sma_20_MAX_60_vwap"],
        "weights": {"values": {"ob_imbalance_l5_sma_20_Ts_Max_5_sma_20_MAX_60_vwap": 1.0},
                    "signs": {"ob_imbalance_l5_sma_20_Ts_Max_5_sma_20_MAX_60_vwap": -1}},
    },
    # 训练期 IC=-0.0418, ICIR=-0.5709（#6）
    "atr_curr_mom_v3": {
        "expressions": [["BEG", "atr_14", "current_ratio_lf", "Ts_Max_5", "momentum_5", "-2", "MIN", "Ts_Min_5", "Ts_Max_5", "If_Else", "Ts_Mean_20", "SEP"]],
        "selected": ["atr_14_current_ratio_lf_Ts_Max_5_momentum_5_-2_MIN"],
        "weights": {"values": {"atr_14_current_ratio_lf_Ts_Max_5_momentum_5_-2_MIN": 1.0},
                    "signs": {"atr_14_current_ratio_lf_Ts_Max_5_momentum_5_-2_MIN": -1}},
    },
    # 训练期 IC=-0.0246, ICIR=-0.5182
    "floatcap_qib_obimb_v3": {
        "expressions": [["BEG", "float_market_cap", "quote_intensity_bid", "Ts_Mean_10", "ob_imbalance_l5", "sma_20", "SIGN", "Cs_Rank", "ADD", "DIV", "SUB", "SEP"]],
        "selected": ["float_market_cap_quote_intensity_bid_Ts_Mean_10_ob"],
        "weights": {"values": {"float_market_cap_quote_intensity_bid_Ts_Mean_10_ob": 1.0},
                    "signs": {"float_market_cap_quote_intensity_bid_Ts_Mean_10_ob": -1}},
    },
    # 训练期 IC=-0.0291, ICIR=-0.5182
    "rev_pe_std_v3": {
        "expressions": [["BEG", "reversal_5", "pe_ttm", "Ts_Std_20", "Ts_Mean_20", "Ts_Mean_10", "ema_20", "rsi_12", "MUL", "SUB", "MUL", "SEP"]],
        "selected": ["reversal_5_pe_ttm_Ts_Std_20_Ts_Mean_20_Ts_Mean_10_"],
        "weights": {"values": {"reversal_5_pe_ttm_Ts_Std_20_Ts_Mean_20_Ts_Mean_10_": 1.0},
                    "signs": {"reversal_5_pe_ttm_Ts_Std_20_Ts_Mean_20_Ts_Mean_10_": -1}},
    },
    # 训练期 IC=0.0228, ICIR=0.4691
    "listdays_roa_intraday_v3": {
        "expressions": [["BEG", "list_days", "roa_avg_ttm", "ema_20", "intraday_return", "Cs_Rank", "5", "MUL", "MUL", "MAX", "SUB", "SEP"]],
        "selected": ["list_days_roa_avg_ttm_ema_20_intraday_return_Cs_Ra"],
        "weights": {"values": {"list_days_roa_avg_ttm_ema_20_intraday_return_Cs_Ra": 1.0},
                    "signs": {"list_days_roa_avg_ttm_ema_20_intraday_return_Cs_Ra": 1}},
    },
    # 训练期 IC=-0.0236, ICIR=-0.4647
    "sma_netbuy_afternoon_v3": {
        "expressions": [["BEG", "sma_20", "Ts_Sum_5", "net_active_buy_amount_main", "afternoon_return", "macd_dea_12_26_9", "gross_profit_rate_ttm", "MAX", "MUL", "MAX", "SUB", "SEP"]],
        "selected": ["sma_20_Ts_Sum_5_net_active_buy_amount_main_afterno"],
        "weights": {"values": {"sma_20_Ts_Sum_5_net_active_buy_amount_main_afterno": 1.0},
                    "signs": {"sma_20_Ts_Sum_5_net_active_buy_amount_main_afterno": -1}},
    },
    # 训练期 IC=-0.0205, ICIR=-0.4446
    "sma_spread_ps_roe_v3": {
        "expressions": [["BEG", "sma_20", "spread_l1_mean", "ps_ttm", "roe_avg_ttm", "MUL", "10", "If_Else", "net_active_buy_amount_main", "MAX", "SUB", "SEP"]],
        "selected": ["sma_20_spread_l1_mean_ps_ttm_roe_avg_ttm_MUL_10_If"],
        "weights": {"values": {"sma_20_spread_l1_mean_ps_ttm_roe_avg_ttm_MUL_10_If": 1.0},
                    "signs": {"sma_20_spread_l1_mean_ps_ttm_roe_avg_ttm_MUL_10_If": -1}},
    },
    # 训练期 IC=-0.0224, ICIR=-0.4332
    "ema_vwap_kdj_macd_v3": {
        "expressions": [["BEG", "ema_20", "vwap_dev", "kdj_k_9_3_3", "vwap_dev", "macd_dea_12_26_9", "Ts_Min_5", "DIV", "MAX", "DIV", "SUB", "SEP"]],
        "selected": ["ema_20_vwap_dev_kdj_k_9_3_3_vwap_dev_macd_dea_12_2"],
        "weights": {"values": {"ema_20_vwap_dev_kdj_k_9_3_3_vwap_dev_macd_dea_12_2": 1.0},
                    "signs": {"ema_20_vwap_dev_kdj_k_9_3_3_vwap_dev_macd_dea_12_2": -1}},
    },
    # 训练期 IC=-0.0206, ICIR=-0.4139（恒等式，与 sma60_floatcap_v2 同 ICIR 的巧合）
    "sma_20": {
        "expressions": [["BEG", "sma_20", "SEP"]],
        "selected": ["sma_20"],
        "weights": {"values": {"sma_20": 1.0}, "signs": {"sma_20": -1}},
    },
    # 训练期 IC=-0.0205, ICIR=-0.4125
    "sma_max_mcap_vwap_v3": {
        "expressions": [["BEG", "sma_20", "Ts_Max_5", "total_market_cap", "20", "vwap_dev", "20", "MIN", "MAX", "MAX", "MIN", "SEP"]],
        "selected": ["sma_20_Ts_Max_5_total_market_cap_20_vwap_dev_20_MI"],
        "weights": {"values": {"sma_20_Ts_Max_5_total_market_cap_20_vwap_dev_20_MI": 1.0},
                    "signs": {"sma_20_Ts_Max_5_total_market_cap_20_vwap_dev_20_MI": -1}},
    },
    # 训练期 IC=-0.0205, ICIR=-0.4117
    "ema_rev_afternoon_kdj_v3": {
        "expressions": [["BEG", "ema_20", "reversal_5", "10", "afternoon_return", "kdj_k_9_3_3", "LOG", "ADD", "MAX", "MAX", "DIV", "SEP"]],
        "selected": ["ema_20_reversal_5_10_afternoon_return_kdj_k_9_3_3_"],
        "weights": {"values": {"ema_20_reversal_5_10_afternoon_return_kdj_k_9_3_3_": 1.0},
                    "signs": {"ema_20_reversal_5_10_afternoon_return_kdj_k_9_3_3_": -1}},
    },
    # ---- 2026-07-28 n1500 池合成 v3（验证集 IC=0.0513, ICIR=0.581, 夏普=+0.68, stress=0.538；
    #      最强合成但 B 项受成员共线压制（v2 实证 0.231），仅作 A 项彩票，优先级低于全部 v3 单因子）----
    "composite_8factor_v3": {
        "expressions": [
            ["BEG", "atr_14", "atr_14", "20", "60", "netflow_amount_main", "DIV", "Ts_Max_5", "SUB", "MUL", "MIN", "SEP"],
            ["BEG", "sma_20", "Ts_Max_5", "Ts_Min_5", "kdj_d_9_3_3", "kdj_d_9_3_3", "float_market_cap", "MIN", "Ts_Zscore_20", "MIN", "SUB", "SEP"],
            ["BEG", "ema_20", "vwap_dev", "netflow_amount_main", "total_market_cap", "pe_ttm", "DIV", "SIGN", "MIN", "If_Else", "Ts_Std_5", "SEP"],
            ["BEG", "net_profit_rate_ttm", "3", "SUB", "ema_20", "MAX", "realized_vol", "reversal_5", "Ts_Mean_20", "DIV", "MAX", "SEP"],
            ["BEG", "float_market_cap", "sma_20", "DIV", "Ts_Std_5", "-2", "current_ratio_lf", "-5", "DIV", "MIN", "DIV", "SEP"],
            ["BEG", "sma_20", "sma_20", "atr_14", "Ts_Rank_20", "ob_imbalance_l5", "Ts_Max_5", "Ts_Min_5", "DIV", "SUB", "MUL", "SEP"],
            ["BEG", "-5", "atr_14", "morning_return", "macd_dea_12_26_9", "Ts_Mean_5", "roe_avg_ttm", "MIN", "MIN", "MAX", "SUB", "SEP"],
            ["BEG", "sma_20", "spread_l1_mean", "ps_ttm", "roe_avg_ttm", "MUL", "10", "If_Else", "net_active_buy_amount_main", "MAX", "SUB", "SEP"],
        ],
        "selected": [
            "atr_14_atr_14_20_60_netflow_amount_main_DIV_Ts_Max",
            "sma_20_Ts_Max_5_Ts_Min_5_kdj_d_9_3_3_kdj_d_9_3_3_f",
            "ema_20_vwap_dev_netflow_amount_main_total_market_c",
            "net_profit_rate_ttm_3_SUB_ema_20_MAX_realized_vol_",
            "float_market_cap_sma_20_DIV_Ts_Std_5_-2_current_ra",
            "sma_20_sma_20_atr_14_Ts_Rank_20_ob_imbalance_l5_Ts",
            "-5_atr_14_morning_return_macd_dea_12_26_9_Ts_Mean_",
            "sma_20_spread_l1_mean_ps_ttm_roe_avg_ttm_MUL_10_If",
        ],
        "weights": {
            "values": {
                "atr_14_atr_14_20_60_netflow_amount_main_DIV_Ts_Max": 0.16842015724124118,
                "sma_20_Ts_Max_5_Ts_Min_5_kdj_d_9_3_3_kdj_d_9_3_3_f": 0.10073710194164266,
                "ema_20_vwap_dev_netflow_amount_main_total_market_c": 0.12311884468330025,
                "net_profit_rate_ttm_3_SUB_ema_20_MAX_realized_vol_": 0.16785673180311847,
                "float_market_cap_sma_20_DIV_Ts_Std_5_-2_current_ra": 0.09566572073531804,
                "sma_20_sma_20_atr_14_Ts_Rank_20_ob_imbalance_l5_Ts": 0.11062557183640699,
                "-5_atr_14_morning_return_macd_dea_12_26_9_Ts_Mean_": 0.124050259409256,
                "sma_20_spread_l1_mean_ps_ttm_roe_avg_ttm_MUL_10_If": 0.1095256123497164,
            },
            "signs": {
                "atr_14_atr_14_20_60_netflow_amount_main_DIV_Ts_Max": 1,
                "sma_20_Ts_Max_5_Ts_Min_5_kdj_d_9_3_3_kdj_d_9_3_3_f": -1,
                "ema_20_vwap_dev_netflow_amount_main_total_market_c": 1,
                "net_profit_rate_ttm_3_SUB_ema_20_MAX_realized_vol_": -1,
                "float_market_cap_sma_20_DIV_Ts_Std_5_-2_current_ra": -1,
                "sma_20_sma_20_atr_14_Ts_Rank_20_ob_imbalance_l5_Ts": 1,
                "-5_atr_14_morning_return_macd_dea_12_26_9_Ts_Mean_": 1,
                "sma_20_spread_l1_mean_ps_ttm_roe_avg_ttm_MUL_10_If": -1,
            },
        },
    },
    # 训练期 IC=-0.0197, ICIR=-0.3884（略低于收编线，n1500 入池背书补收）
    "floatcap_sma_div_v3": {
        "expressions": [["BEG", "float_market_cap", "sma_20", "DIV", "Ts_Std_5", "-2", "current_ratio_lf", "-5", "DIV", "MIN", "DIV", "SEP"]],
        "selected": ["float_market_cap_sma_20_DIV_Ts_Std_5_-2_current_ra"],
        "weights": {"values": {"float_market_cap_sma_20_DIV_Ts_Std_5_-2_current_ra": 1.0},
                    "signs": {"float_market_cap_sma_20_DIV_Ts_Std_5_-2_current_ra": -1}},
    },
    # ---- 2026-07-29 LLM 训练运行入池新面孔（该次合成验证集 IC=0.0546/ICIR=0.657/夏普+1.02/stress 0.621；
    #      池内含冠军 atr_atr_netflow_v3（权重 0.195）+ 已交 sma_spread_ps_roe_v3 / 在册 sma_kdj_float_v2，
    #      合成勿交（稀释冠军与已交单因子），只收编 4 个新单因子）----
    "atr_netflow_obimb_llm": {
        "expressions": [["BEG", "atr_14", "netflow_amount_main", "Ts_Std_20", "ob_imbalance_l5", "list_days", "realized_vol", "DIV", "DIV", "ADD", "MUL", "SEP"]],
        "selected": ["atr_14_netflow_amount_main_Ts_Std_20_ob_imbalance_"],
        "weights": {"values": {"atr_14_netflow_amount_main_Ts_Std_20_ob_imbalance_": 1.0},
                    "signs": {"atr_14_netflow_amount_main_Ts_Std_20_ob_imbalance_": -1}},
    },
    "floatcap_vol_morning_llm": {
        "expressions": [["BEG", "float_market_cap", "total_market_cap", "Cs_Rank", "volatility_5", "morning_return", "ema_20", "MAX", "ADD", "ADD", "MUL", "SEP"]],
        "selected": ["float_market_cap_total_market_cap_Cs_Rank_volatili"],
        "weights": {"values": {"float_market_cap_total_market_cap_Cs_Rank_volatili": 1.0},
                    "signs": {"float_market_cap_total_market_cap_Cs_Rank_volatili": -1}},
    },
    "pb_vwap_float_llm": {
        "expressions": [["BEG", "pb", "sma_20", "float_market_cap", "vwap_dev", "pb", "Ts_Mean_10", "If_Else", "DIV", "MAX", "ABS", "SEP"]],
        "selected": ["pb_sma_20_float_market_cap_vwap_dev_pb_Ts_Mean_10_"],
        "weights": {"values": {"pb_sma_20_float_market_cap_vwap_dev_pb_Ts_Mean_10_": 1.0},
                    "signs": {"pb_sma_20_float_market_cap_vwap_dev_pb_Ts_Mean_10_": -1}},
    },
    "roa_kdj_atr_llm": {
        "expressions": [["BEG", "roa_avg_ttm", "kdj_d_9_3_3", "ABS", "Ts_Sum_5", "3", "atr_14", "MUL", "LOG", "MUL", "ADD", "SEP"]],
        "selected": ["roa_avg_ttm_kdj_d_9_3_3_ABS_Ts_Sum_5_3_atr_14_MUL_"],
        "weights": {"values": {"roa_avg_ttm_kdj_d_9_3_3_ABS_Ts_Sum_5_3_atr_14_MUL_": 1.0},
                    "signs": {"roa_avg_ttm_kdj_d_9_3_3_ABS_Ts_Sum_5_3_atr_14_MUL_": -1}},
    },
    # ---- 2026-07-29 LLM 候选表补收（第二批，ICIR>=0.50 新面孔）----
    # 训练期 IC=-0.0506, ICIR=-0.6002（全表 #5，netbuy 资金流+早盘+roa 混合）
    "netbuy_abs_morning_llm": {
        "expressions": [["BEG", "sma_20", "net_active_buy_amount_main", "ABS", "morning_return", "reversal_5", "roa_avg_ttm", "SUB", "MUL", "If_Else", "SEP"]],
        "selected": ["sma_20_net_active_buy_amount_main_ABS_morning_retu"],
        "weights": {"values": {"sma_20_net_active_buy_amount_main_ABS_morning_retu": 1.0},
                    "signs": {"sma_20_net_active_buy_amount_main_ABS_morning_retu": -1}},
    },
    # 训练期 IC=-0.0614（全表 |IC| 冠军）, ICIR=-0.5435（qia 家族新表达：ask 强度+kdj+rvol）
    "qia_kdj_rvol_llm": {
        "expressions": [["BEG", "quote_intensity_ask", "ema_20", "DIV", "kdj_d_9_3_3", "realized_vol", "Ts_Mean_20", "kdj_d_9_3_3", "DIV", "MIN", "ADD", "SEP"]],
        "selected": ["quote_intensity_ask_ema_20_DIV_kdj_d_9_3_3_realize"],
        "weights": {"values": {"quote_intensity_ask_ema_20_DIV_kdj_d_9_3_3_realize": 1.0},
                    "signs": {"quote_intensity_ask_ema_20_DIV_kdj_d_9_3_3_realize": -1}},
    },
    # 训练期 IC=-0.0296, ICIR=-0.5070（spread 流动性+早盘+current_ratio 财务）
    "spread_morning_curr_llm": {
        "expressions": [["BEG", "sma_20", "Ts_Max_5", "spread_l1_mean", "morning_return", "current_ratio_lf", "Cs_Rank", "MAX", "ADD", "Ts_Rank_5", "MUL", "SEP"]],
        "selected": ["sma_20_Ts_Max_5_spread_l1_mean_morning_return_curr"],
        "weights": {"values": {"sma_20_Ts_Max_5_spread_l1_mean_morning_return_curr": 1.0},
                    "signs": {"sma_20_Ts_Max_5_spread_l1_mean_morning_return_curr": -1}},
    },
}

FROZEN_CONFIG_NAME = "qia_kdj_rvol_llm"  # 本次提交使用的配置键；None = 回训练模式
FROZEN_CONFIG = FROZEN_CONFIGS[FROZEN_CONFIG_NAME] if FROZEN_CONFIG_NAME else None


# ================= All-in-One 主流程 =================

def _work_data_dir() -> Path:
    """平台工作目录下的 data/（步骤 9 同款定位逻辑），供各阶段结果落盘。"""
    root = Path.cwd()
    if not (root / "data").exists():
        root = root.parent
    d = root / "data"
    d.mkdir(parents=True, exist_ok=True)
    return d


def run_all_in_one(
    start_date: str = "2019-01-01",
    end_date: str = "2024-12-31",
    train_end: str = "2023-12-31",
    n_random: int = 500,
    use_llm: bool = True,
    bar1m_table: str = "bigalpha_2026_stock_bar1m",
) -> pd.Series:
    """
    端到端流程：
    1. 加载平台日频因子库 + 1 分钟行情库构造的高频日频特征；
    2. 合并日频因子与高频日频特征；
    3. 生成候选表达式（随机 + 经验 + 可选 LLM），排除原始日频价格/成交字段；
    4. 评估候选因子 → IC 阈值粗筛 → 相关性去重 → 增量 ICIR 入池（池子级）→ ICIR 加权合成；
    5. 返回最终 factor Series（index=date/instrument）。
    """
    # 1. 加载数据
    print("获取成分股...")
    inst_df = get_csi1000_instruments([start_date, end_date])
    instruments = inst_df["instrument"].unique().tolist()
    print(f"历史成分股数（去重）: {len(instruments)}")

    print("加载日频因子库...")
    daily_df = load_factor_library(start_date, end_date, instruments, constituents_df=inst_df)
    print(f"日频数据 shape: {daily_df.shape}")
    if daily_df.empty:
        print("日频数据为空，结束")
        return pd.Series(dtype=float)
    daily_counts = daily_df.groupby("date")["instrument"].nunique()
    print(f"每日成分股数量: min={daily_counts.min()}, max={daily_counts.max()}, mean={daily_counts.mean():.1f}")

    # 根据实际数据自动调整训练/验证划分
    actual_start = pd.to_datetime(daily_df["date"].min())
    actual_end = pd.to_datetime(daily_df["date"].max())
    print(f"实际数据区间: {actual_start.date()} ~ {actual_end.date()}")
    train_end_dt = pd.to_datetime(train_end)
    if train_end_dt >= actual_end:
        total_days = (actual_end - actual_start).days
        val_days = min(365, max(60, int(total_days * 0.2)))
        train_end_dt = actual_end - pd.Timedelta(days=val_days)
        train_end = train_end_dt.strftime("%Y-%m-%d")
        print(f"训练截止自动调整为: {train_end}")

    daily_df = prepare_panel(daily_df)
    returns = compute_forward_returns(daily_df["close"])
    # 边界标签修正：最后一个训练日的下期收益 = 测试期首日收盘价 / 当日收盘价 - 1，
    # 会把测试期信息渗入训练（IC 粗筛/入池/权重拟合）。将该日标签置 NaN，
    # 之后所有 returns.loc[train_dates] 切片在 compute_ic 的 dropna 中自动剔除它；
    # 验证集打分只用 test_dates（> train_end），不受此 NaN 影响。
    _trade_dates = returns.index.get_level_values("date").unique().sort_values()
    _train_side = _trade_dates[_trade_dates <= train_end]
    if len(_train_side) > 0:
        _boundary_date = _train_side[-1]
        returns[returns.index.get_level_values("date") == _boundary_date] = np.nan
        print(f"边界标签修正：{_boundary_date} 的下期收益用到测试期首日收盘价，已置 NaN")
    del _trade_dates, _train_side

    # 2. 从 1 分钟行情库构造高频日频特征，并和日频因子库合并
    print("准备高频日频特征...")
    # 已移除 parquet 缓存逻辑：平台会复用 work 目录，旧缓存（字段口径与现版本不一致）
    # 会被无限期加载，存在隐患；且 16 字段精简查询后全量构建耗时可接受，每次强制重新构建
    # print("从 1 分钟行情库构造高频日频特征（按月分块）...")
    hf_df = build_hf_daily_features(start_date, end_date, instruments=instruments, chunk_size=500, verbose=True, bar1m_table=bar1m_table)
    print(f"高频特征 shape: {hf_df.shape}")
    if hf_df.empty:
        print("无高频特征，仅使用日频因子库")
        merged_df = daily_df.reset_index()
    else:
        merged_df = merge_daily_and_hf(daily_df.reset_index(), hf_df)
    print(f"合并后数据 shape: {merged_df.shape}")

    # 预置 (date, instrument) 索引，避免每个候选表达式在 evaluate 内重复 set_index/sort_index
    merged_df = merged_df.set_index(["date", "instrument"]).sort_index()

    # ===== 冻结配置模式：跳过步骤 3~7（候选生成/评估/筛选/权重拟合） =====
    if FROZEN_CONFIG is not None:
        print(f"冻结配置模式[{FROZEN_CONFIG_NAME}]：{len(FROZEN_CONFIG['selected'])} 个固化因子，跳过训练/筛选")
        all_dates = merged_df.index.get_level_values("date").unique().sort_values()
        test_dates = all_dates[all_dates > train_end]
        _ev = RPNEvaluator(VOCAB)
        _vals = FROZEN_CONFIG["weights"]["values"]
        _sgns = FROZEN_CONFIG["weights"]["signs"]
        final_factor = None
        for _name, _expr in zip(FROZEN_CONFIG["selected"], FROZEN_CONFIG["expressions"]):
            _fc = _ev.evaluate(_expr, merged_df)
            if _fc is None or _fc.dropna().empty:
                raise ValueError(f"冻结表达式求值失败: {_name}")
            # 与训练模式同一套合成：逐因子截面去极值+zscore 后按固化权重/符号加权
            _contrib = preprocess_factor(_fc).fillna(0) * _sgns[_name] * _vals[_name]
            final_factor = _contrib if final_factor is None else final_factor + _contrib
        # 验证集打分仅观测、不回灌任何决策
        _score = None
        try:
            _style_cols = ["total_market_cap", "beta_000300SH_22", "momentum_5", "volatility_5"]
            _score = competition_score(
                final_factor.loc[test_dates], returns.loc[test_dates],
                styles=daily_df[_style_cols], style_cols=_style_cols,
                stress_mask=make_stress_mask(test_dates),
            )
            print("验证集得分:", _score)
        except Exception as _e:
            import traceback
            traceback.print_exc()
            print(f"验证集打分失败: {_e}")
        # 结果落盘：平台日志查看器会丢弃 print 行，因子值与得分以工作目录文件为准；
        # 文件名带配置键，平台复用 work 目录时多次提交的结果互不覆盖；
        # submit_config.json 记录本次运行实际使用的配置，供核对（昨日起它只在训练路径落盘）
        _out = final_factor.loc[test_dates].fillna(0)
        _dir = _work_data_dir()
        _tag = FROZEN_CONFIG_NAME or "adhoc"
        _out.rename("factor").reset_index().to_csv(_dir / f"submit_factor_{_tag}.csv", index=False)
        with open(_dir / f"frozen_score_{_tag}.json", "w", encoding="utf-8") as _f:
            json.dump({"config": _tag, "score": _score}, _f, ensure_ascii=False, indent=2)
        with open(_dir / "submit_config.json", "w", encoding="utf-8") as _f:
            json.dump({"config": _tag, **FROZEN_CONFIG}, _f, ensure_ascii=False, indent=2)
        print(f"已保存 submit_factor_{_tag}.csv / frozen_score_{_tag}.json / submit_config.json")
        return _out

    # 3. 构造词表：允许日频非原始因子 + 高频日频特征，排除原始日频价格/成交量字段
    RAW_DAILY_FIELDS = {
        "close", "volume", "amount", "turn", "change_ratio", "daily_return",
    }
    available_fields = [
        c for c in (VOCAB.BASE_FIELDS + VOCAB.HF_FIELDS)
        if c in merged_df.columns and c not in RAW_DAILY_FIELDS
    ]
    print(f"可用字段数（排除原始日频字段后）: {len(available_fields)}")
    # 候选评估面板全为日频（日频因子 + 高频日频特征），分钟算子必然求值失败（曾 490/507 候选出生即死），词表直接剔除
    use_vocab = Vocabulary(field_subset=available_fields, include_minute_ops=False)

    def _expr_fields_ok(expr):
        return all(
            tok in available_fields or tok in use_vocab.constants or tok in use_vocab.operators or tok in ("BEG", "SEP")
            for tok in expr
        )

    # 4. 生成候选
    print("生成候选表达式...")
    candidates = generate_random_candidates(use_vocab, n=n_random, max_len=12)

    handcrafted = [
        ["BEG", "momentum_5", "Ts_Mean_5", "SUB", "SEP"],
        ["BEG", "netflow_amount_rate_main", "Cs_Zscore", "SEP"],
        ["BEG", "turn", "turn", "Ts_Mean_20", "DIV", "LOG", "SEP"],
        ["BEG", "pb", "Cs_Rank", "NEG", "pe_ttm", "Cs_Rank", "ADD", "SEP"],
        ["BEG", "realized_vol", "Cs_Rank", "NEG", "SEP"],
        ["BEG", "intraday_return", "Cs_Zscore", "SEP"],
        ["BEG", "ob_imbalance_l5", "morning_return", "MUL", "Cs_Rank", "SEP"],
    ]
    candidates += [e for e in handcrafted if _expr_fields_ok(e)]

    llm_status = {"use_llm": use_llm, "api_key": ("***" + os.environ.get("LLM_API_KEY", "")[-4:]) if os.environ.get("LLM_API_KEY") else None,
                  "status": "skipped", "n_expressions": 0, "error": None}
    print("LLM_API_KEY:", "***" + os.environ.get("LLM_API_KEY", "")[-4:] if os.environ.get("LLM_API_KEY") else None, flush=True)
    if use_llm and os.environ.get("LLM_API_KEY"):
        try:
            print("调用 LLM 生成候选表达式...", flush=True)
            llm_exprs = offline_llm_generate(
                api_key=os.environ.get("LLM_API_KEY"),
                base_url=os.environ.get("LLM_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1"),
                model=os.environ.get("LLM_MODEL", "glm-5.2"),
                n=int(os.environ.get("LLM_N", "20")),
                json_mode=os.environ.get("LLM_JSON_MODE", "1").lower() in ("1", "true", "yes"),
            )
            llm_exprs = [e for e in llm_exprs if _expr_fields_ok(e)]
            print(f"有效 LLM 表达式: {len(llm_exprs)}", flush=True)
            candidates += llm_exprs
            llm_status.update(status="ok", n_expressions=len(llm_exprs))
        except Exception as e:
            print(f"LLM 调用失败，跳过: {e}", flush=True)
            llm_status.update(status="error", error=str(e))
    else:
        print("未配置 LLM_API_KEY，跳过 LLM 生成", flush=True)

    # 5. 评估候选因子
    print("评估候选因子...")
    # 只保留候选表达式实际引用的列再送入并行评估：
    # loky 每个 batch 都要把 df 序列化/memmap 给 worker，裁掉无关列可显著降低传输开销。
    # 候选 token 已在生成阶段限定在 available_fields 内，裁剪不改变任何评估结果。
    used_cols = sorted({tok for expr in candidates for tok in expr if tok in merged_df.columns})
    print(f"候选表达式实际引用列数: {len(used_cols)} / {len(merged_df.columns)}")
    eval_df = merged_df[used_cols]
    # 候选初筛/截断只能使用训练区间标签：测试期标签置 NaN 后，IC 计算自动 dropna，
    # 避免用提交区间的 ICIR 挑选因子（选择偏差型泄露）。因子序列本身不含标签，
    # 仍在全区间求值，不影响 2024 提交区间的因子输出。
    eval_returns = returns.copy()
    eval_returns[eval_returns.index.get_level_values("date") > train_end] = np.nan
    factors_df, name_to_expr = evaluate_candidates(candidates, eval_df, eval_returns)
    print(f"有效因子数: {len(factors_df.columns)}")
    if factors_df.empty:
        print("无有效因子，结束")
        return pd.Series(dtype=float)

    # 6. 划分训练/验证
    all_dates = factors_df.index.get_level_values("date").unique().sort_values()
    train_dates = all_dates[all_dates <= train_end]
    test_dates = all_dates[all_dates > train_end]
    print(f"训练日期数: {len(train_dates)}, 验证日期数: {len(test_dates)}")

    # 6.4 IC 阈值粗筛（完整版）：训练期 |IC| >= 0.005 且 |ICIR| >= 0.05，双阈值同时满足才保留
    # 注意 ICIR 口径是日度 IC 的 mean/std，不是年化：日度 0.3 ≈ 年化 4.7，几乎无因子能通过；
    # 日度 0.1 ≈ 年化 1.6（中上）、0.05 ≈ 年化 0.8（池子入场券的宽松口径）。
    # 阈值迭代记录：ICIR 0.3 → 只剩 2 个原始字段；0.1 → 仍 2 个；0.05 + IC 0.01 →
    # 8 个 ICIR 0.14~0.17 的因子被 |IC|>=0.01 卡掉（IC 仅 0.006~0.008），故 IC 也放宽到 0.005。
    print("IC 粗筛诊断（训练期，按 |ICIR| 降序）：")
    _diag = []
    for c in factors_df.columns:
        _m = compute_ic_metrics(compute_ic(factors_df.loc[train_dates, c], returns.loc[train_dates]))
        _diag.append((c, _m["ic_mean"], _m["icir"]))
    for c, _ic, _icir in sorted(_diag, key=lambda t: -(abs(t[2]) if not pd.isna(t[2]) else 0)):
        print(f"  {c}: IC={_ic:.4f}, ICIR={_icir:.4f}")
    # 全量候选 IC 表落盘：因子名被截断到 50 字符、表达式不可从名字复原，
    # 必须连同完整 RPN 表达式一起存，供多提交矩阵挑选单因子配置（2026-07-28）
    try:
        _cand_table = [
            {"name": c, "expr": name_to_expr.get(c),
             "ic": None if pd.isna(_ic) else float(_ic),
             "icir": None if pd.isna(_icir) else float(_icir)}
            for c, _ic, _icir in _diag
        ]
        with open(_work_data_dir() / "candidate_ic_table.json", "w", encoding="utf-8") as _f:
            json.dump(_cand_table, _f, ensure_ascii=False, indent=2)
        print(f"候选 IC 表已落盘 candidate_ic_table.json（{len(_cand_table)} 条）")
    except Exception as _e:
        print(f"candidate_ic_table 落盘失败: {_e}")
    del _diag
    coarse = ic_threshold_select(
        factors_df.loc[train_dates], returns.loc[train_dates],
        min_abs_ic=0.01, min_icir=0.05,
    )
    dropped_ic = [c for c in factors_df.columns if c not in coarse]
    if dropped_ic:
        print(f"IC 阈值粗筛（训练期 |IC|>=0.005 且 |ICIR|>=0.05）：移除 {len(dropped_ic)} 个 -> {dropped_ic}")
        factors_df = factors_df[coarse]
        name_to_expr = {k: v for k, v in name_to_expr.items() if k in coarse}
    if factors_df.empty:
        print("IC 阈值粗筛后无有效因子，结束")
        return pd.Series(dtype=float)

    # 6.5 相关性去重：高相关因子只保留训练期 |ICIR| 最高者，避免同一信号（如动量/反转镜像）
    # 被重复计入加权和。相关性用训练期日度 rank 相关计算，不触碰测试期数据。
    CORR_THRESHOLD = 0.8
    train_fac = factors_df.loc[train_dates]
    icir_abs = {}
    for c in factors_df.columns:
        _m = compute_ic_metrics(compute_ic(train_fac[c], returns.loc[train_dates]))
        icir_abs[c] = 0.0 if pd.isna(_m["icir"]) else abs(_m["icir"])
    ordered = sorted(factors_df.columns, key=lambda c: icir_abs[c], reverse=True)
    corr_mat = train_fac.groupby(level="date", group_keys=False).rank(pct=True).corr().abs()
    keep = []
    for c in ordered:
        if keep and corr_mat.loc[c, keep].max() >= CORR_THRESHOLD:
            continue
        keep.append(c)
    dropped = [c for c in factors_df.columns if c not in keep]
    if dropped:
        print(f"相关性去重（训练期 |corr| >= {CORR_THRESHOLD}）：移除 {len(dropped)} 个 -> {dropped}")
        factors_df = factors_df[keep]
        name_to_expr = {k: v for k, v in name_to_expr.items() if k in keep}

    # 7. 池子级评估：贪婪增量 ICIR 入池（合并 ICIR 增量 >= 0.005 才入池），再对入池因子做 ICIR 加权合成
    print("增量 ICIR 入池（池子级评估，min_improvement=0.005）...")
    selected = incremental_pool_select(
        factors_df, returns, train_dates=train_dates,
        corr_threshold=CORR_THRESHOLD, max_factors=10, min_improvement=0.005,
    )
    if not selected:
        print("警告：增量入池结果为空，回退为保留去重后的全部因子")
        selected = list(factors_df.columns)
    dropped_pool = [c for c in factors_df.columns if c not in selected]
    if dropped_pool:
        print(f"增量入池：移除 {len(dropped_pool)} 个 -> {dropped_pool}")
    print(f"最终入池 {len(selected)} 个 -> {selected}")
    factors_df = factors_df[selected]
    name_to_expr = {k: v for k, v in name_to_expr.items() if k in selected}
    print("ICIR 加权合成入池因子...")
    # 权重/符号只能用训练区间标签拟合；若用 test_dates，提交区间的未来收益会泄露进因子取值
    train_ret = returns.loc[train_dates]
    weights = {}
    signs = {}
    for c in factors_df.columns:
        ic = compute_ic(factors_df[c], train_ret)
        metrics = compute_ic_metrics(ic)
        icir = metrics.get("icir", 0)
        ic_mean = metrics.get("ic_mean", 0)
        weights[c] = 0.0 if pd.isna(icir) else abs(icir)
        signs[c] = -1 if ic_mean < 0 else 1
    total_w = sum(abs(w) for w in weights.values())
    if total_w > 1e-9:
        weights = {c: w / total_w for c, w in weights.items()}
    else:
        weights = {c: 1.0 / len(factors_df.columns) for c in factors_df.columns}

    final_factor = None
    for c in factors_df.columns:
        # 合成前对每个因子做截面去极值+zscore：统一量纲后再按 ICIR 权重加权，
        # 避免 amount/volume 类大尺度原始因子主导加权和、拖垮多空两端。
        # 逐日 winsorize+zscore 是截面单调变换，不改变各因子 Rank IC，权重/符号依然有效；
        # 且预处理对象是单个因子而非最终合成值，平台对最终因子仍只做一次预处理，无重复处理。
        fc = preprocess_factor(factors_df[c])
        contrib = fc.fillna(0) * signs[c] * weights[c]
        final_factor = contrib if final_factor is None else final_factor + contrib

    # 8. 在验证集上打分
    test_factor = final_factor.loc[test_dates]
    test_returns = returns.loc[test_dates]

    # 8.1 多空夏普诊断
    diagnose_long_short_sharpe(test_factor, test_returns, label="最终合成因子")

    try:
        style_cols = ["total_market_cap", "beta_000300SH_22", "momentum_5", "volatility_5"]
        missing_cols = [c for c in style_cols if c not in daily_df.columns]
        if missing_cols:
            raise KeyError(f"日频因子库缺少风格字段: {missing_cols}，可用字段: {list(daily_df.columns)}")
        styles = daily_df[style_cols]
        stress_mask = make_stress_mask(test_dates)
        test_score = competition_score(
            test_factor,
            test_returns,
            styles=styles,
            style_cols=style_cols,
            stress_mask=stress_mask,
        )
        print("验证集得分:", test_score)
    except Exception as e:
        import traceback

        traceback.print_exc()  # 打出完整堆栈，避免评分被静默吞掉后无从排查
        print(f"验证集打分失败: {e}")

    # 9. 保存配置
    project_root = Path.cwd()
    if not (project_root / "data").exists():
        project_root = project_root.parent
    data_dir = project_root / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    submit_path = data_dir / "submit_config.json"
    output = {
        "expressions": [name_to_expr[c] for c in factors_df.columns if c in name_to_expr],
        "selected": list(factors_df.columns),
        "llm": llm_status,
        "weights": {
            "method": "ic_weighted_all",
            "values": weights,
            "signs": signs,
        },
    }
    with open(submit_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"已保存 {submit_path}")
    # 初筛淘汰统计落盘：平台日志查看器会丢弃 print 行，诊断信息以文件形式取回
    stats_path = data_dir / "eval_fail_stats.json"
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump({"fail_stats": EVAL_FAIL_STATS, "fail_examples": EVAL_FAIL_MSGS},
                  f, ensure_ascii=False, indent=2)
    print(f"已保存 {stats_path}")

    # 10. 返回原始因子值（平台会自己做去极值/标准化，避免双重处理）
    test_factor = test_factor.fillna(0)
    return test_factor



# ================= main：比赛提交入口 =================
def main(datasources, start_date, end_date):
    """
    因子构建主函数。评测时平台会自动替换 datasources / start_date / end_date
    三个入参并调用本函数（签名必须与平台模板一致，否则时间参数无法注入）。

    参数:
        datasources (dict): 数据源表名映射 {逻辑名: 物理表名}，"bar1m" 为分钟 K 线表。
                            公榜/私榜物理表名由平台切换，SQL 中不得硬编码表名。
        start_date (str): 提交区间开始时间（公榜为 2024-01-01）
        end_date (str):   提交区间结束时间（公榜为 2024-12-31）

    返回:
        pd.DataFrame: ['date', 'instrument', 'factor']，不含 inf
    """
    # 分钟 K 线物理表名由平台注入；本地自测时回退到公榜表名
    bar1m_table = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")

    # 平台注入的 start_date/end_date 是提交区间（公榜为 2024 全年）；
    # 数据源保留 2019 年至今的数据，训练集固定从 2019-01-01 开始，
    # 训练截止 = 提交区间开始前一天（公榜即 2023-12-31）
    end_date = pd.to_datetime(end_date).strftime("%Y-%m-%d")
    train_start = "2019-01-01"
    train_end = (pd.to_datetime(start_date) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    try:
        final_factor = run_all_in_one(
            start_date=train_start,
            end_date=end_date,
            train_end=train_end,
            bar1m_table=bar1m_table,
        )
    except Exception:
        import traceback

        traceback.print_exc()  # 打到 stderr，确保平台日志面板能看到错误
        raise

    if final_factor.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    result = final_factor.reset_index()
    result.columns = ["date", "instrument", "factor"]
    return result

if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时自行构造数据源映射（评测时由平台注入，逻辑名固定为 "bar1m"/"financial"）
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)
    print(factor_data.head())
    print("shape:", factor_data.shape)
    try:
        # 平台日志查看器会吞 print 的 stdout，display 的 HTML 表格输出到 cell 结果区，更可靠
        from IPython.display import display
        display(factor_data.head(20))
    except Exception:
        pass

    # 读取平台因子库用于回归评估，您可以换成自己的因子库
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统：
    # process_pools=False 表示不对因子库再做预处理（bigalpha_2026_factorlib 已处理过）
    # show=True 表示画出评估图表
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=False,
    )

[2026-07-29 20:27:11] [info     ] 计算因子，区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
获取成分股...
历史成分股数（去重）: 2075
加载日频因子库...
  DEBUG 2019Q1: insts=1000, part_shape=(58000, 38)
  DEBUG 2019Q2: insts=1100, part_shape=(60000, 38)
  DEBUG 2019Q3: insts=1000, part_shape=(65000, 38)
  DEBUG 2019Q4: insts=1100, part_shape=(61000, 38)
  DEBUG 2020Q1: insts=1003, part_shape=(58000, 38)
  DEBUG 2020Q2: insts=1100, part_shape=(59000, 38)
  DEBUG 2020Q3: insts=1006, part_shape=(66000, 38)
  DEBUG 2020Q4: insts=1101, part_shape=(60000, 38)
  DEBUG 2021Q1: insts=1003, part_shape=(58000, 38)
  DEBUG 2021Q2: insts=1100, part_shape=(60000, 38)
  DEBUG 2021Q3: insts=1002, part_shape=(64000, 38)
  DEBUG 2021Q4: insts=1101, part_shape=(61000, 38)
  DEBUG 2022Q1: insts=1001, part_shape=(58000, 38)
  DEBUG 2022Q2: insts=1100, part_shape=(59000, 38)
  DEBUG 2022Q3: insts=1000, part_shape=(65000, 38)
  DEBUG 2022Q4: insts=1100, part_shape=(60000, 38)
  DEBUG 2023Q1: insts=1000, part_shape=(59000, 38)
  DEBUG 2023

构建高频特征:   0%|          | 0/360 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:    8.3s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   11.7s
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:   21.1s
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:   27.4s
[Parallel(n_jobs=-1)]: Done  33 tasks      | elapsed:   37.6s
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   48.6s
[Parallel(n_jobs=-1)]: Done  53 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done  77 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done  90 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 105 tasks      | elapsed:  2.0min
[Parallel(n_jobs=-1)]: Done 120 tasks      | elapsed:  2.3min
[Parallel(n_jobs=-1)]: Done 137 tasks      | elapsed:  2.7min
[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed:  3.0min
[Parallel(n_jobs=-1)]: Done 173 tasks      | elapsed:  3

高频特征 shape: (2736416, 17)
合并后数据 shape: (1456000, 53)
冻结配置模式[qia_kdj_rvol_llm]：1 个固化因子，跳过训练/筛选
验证集得分: {'ic_mean': 0.05445789610937787, 'icir': 0.4067395208235823, 'long_short_sharpe': 1.0233798229365556, 'stress_icir': 0.35785450018971965}
已保存 submit_factor_qia_kdj_rvol_llm.csv / frozen_score_qia_kdj_rvol_llm.json / submit_config.json
         date instrument    factor
0  2024-01-02  000006.SZ -1.406438
1  2024-01-02  000012.SZ -1.452369
2  2024-01-02  000016.SZ -2.137720
3  2024-01-02  000019.SZ  0.723948
4  2024-01-02  000025.SZ -0.212429
shape: (242000, 3)


,date,instrument,factor
0,2024-01-02,000006.SZ,-1.406438
1,2024-01-02,000012.SZ,-1.452369
2,2024-01-02,000016.SZ,-2.137720
3,2024-01-02,000019.SZ,0.723948
4,2024-01-02,000025.SZ,-0.212429
5,2024-01-02,000028.SZ,-0.475214
6,2024-01-02,000029.SZ,-0.135788
7,2024-01-02,000030.SZ,0.910055
8,2024-01-02,000032.SZ,-1.323726
9,2024-01-02,000034.SZ,-0.768196


[2026-07-29 20:35:02] [info     ] 读取因子库，区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
[2026-07-29 20:35:03] [warning  ] bigalpha_eval._latest version='v4' (use ._latest for dev only, not for prod)
[2026-07-29 20:35:05] [info     ] bigalpha_eval.v4 开始运行 ..
[2026-07-29 20:35:06] [warning  ] 未传入官方评估窗口 start_date/end_date，回退到数据自身范围（仅建议本地调试时使用）
[2026-07-29 20:35:06] [info     ] 对齐中证1000历史成分后，官方评估窗口: 2024-01-02 至 2024-12-31
[2026-07-29 20:35:06] [info     ] ========== 数据检查 ==========
[2026-07-29 20:35:06] [info     ] 通过：列名检查（date/instrument + 至少 1 个因子列） factor_cols=['factor', 'close', 'volume', 'amount', 'turn', 'change_ratio', 'daily_return', 'momentum_5', 'reversal_5', 'volatility_5', 'total_market_cap', 'float_market_cap', 'pe_ttm', 'pb', 'ps_ttm', 'sma_20', 'ema_20', 'macd_diff_12_26_9', 'macd_dea_12_26_9', 'macd_hist_12_26_9', 'rsi_12', 'kdj_k_9_3_3', 'kdj_d_9_3_3', 'bias_20', 'cci_14', 'atr_14', 'roe_avg_ttm', 'roa_avg_ttm', 'gross_profit_rate_ttm', 'net_profit_rate_ttm', 'debt_to_asset